# Forklift Target Anomaly Inference

保存済みの特徴量ベース異常検知モデルを読み込み、複数の前処理済み動画に対して窓別スコアを推論します。学習処理は含めません。

## 1. セットアップ

In [104]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import butter, filtfilt, resample_poly
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

try:
    import librosa
except Exception as exc:
    librosa = None
    warnings.warn(f"librosa could not be imported. Audio features will use a scipy fallback. reason={exc}")

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 120)


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "movie_preprocess").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repository root from {start}.")


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "anomaly_feature_poc"
MODEL_PATH = OUTPUT_DIR / "isolation_forest_feature_poc.joblib"

# 推論とグラフ出力の切り替え。
# まずは特徴量確認だけを行うため、推論と従来グラフはOFFにしています。
RUN_INFERENCE = True
PLOT_EXISTING_GRAPHS = True
PLOT_RAW_FLOW_XY = True
RAW_FLOW_SAMPLE_SEC = 0.1
RAW_FLOW_MAGNITUDE_LOWPASS_CUTOFF_HZ = 1.0
RAW_FLOW_MAGNITUDE_LOWPASS_ORDER = 2
FLOW_SCORE_WINDOW_SEC = 1.0
FLOW_SCORE_HOP_SEC = 0.5
FLOW_SCORE_ALPHA_MIN = 0.04
FLOW_SCORE_ALPHA_MAX = 0.42
FLOW_SCORE_MIN_VISIBLE = 1e-6
FLOW_SCORE_HIGH_RATIO_FRACTION = 0.5
FLOW_SCORE_DISTINCTIVENESS_WEIGHT = 0.85

if MODEL_PATH.exists():
    artifacts = joblib.load(MODEL_PATH)
else:
    if RUN_INFERENCE:
        raise FileNotFoundError(f"Trained model was not found: {MODEL_PATH}")
    warnings.warn(f"Trained model was not found. Raw flow extraction will continue without inference: {MODEL_PATH}")
    artifacts = {}

settings = artifacts.get("settings", {})
FLOW_SCORE_WINDOW_SEC = float(settings.get("flow_score_window_sec", FLOW_SCORE_WINDOW_SEC))
FLOW_SCORE_HOP_SEC = float(settings.get("flow_score_hop_sec", FLOW_SCORE_HOP_SEC))
FLOW_SCORE_ALPHA_MIN = float(settings.get("flow_score_alpha_min", FLOW_SCORE_ALPHA_MIN))
FLOW_SCORE_ALPHA_MAX = float(settings.get("flow_score_alpha_max", FLOW_SCORE_ALPHA_MAX))
FLOW_SCORE_MIN_VISIBLE = float(settings.get("flow_score_min_visible", FLOW_SCORE_MIN_VISIBLE))
FLOW_SCORE_HIGH_RATIO_FRACTION = float(settings.get("flow_score_high_ratio_fraction", FLOW_SCORE_HIGH_RATIO_FRACTION))
# Flow magnitude below this value is treated as directionless for direction-change plots/scores.
# Tune this if low-motion periods still produce noisy direction changes.
FLOW_DIRECTION_MIN_MAG = float(settings.get("flow_direction_min_mag", 0.025))
DIRECTION_JITTER_HIGH_THRESHOLD = float(settings.get("direction_jitter_high_threshold", 1.5))
DIRECTION_JITTER_LOW_THRESHOLD = float(settings.get("direction_jitter_low_threshold", 0.8))
DIRECTION_JITTER_THRESHOLD_MODE = str(settings.get("direction_jitter_threshold_mode", "percentile")).lower()
DIRECTION_JITTER_HIGH_PERCENTILE = float(settings.get("direction_jitter_high_percentile", 85.0))
DIRECTION_JITTER_LOW_PERCENTILE = float(settings.get("direction_jitter_low_percentile", 60.0))
DIRECTION_JITTER_LOW_EXPANSION_STEPS = int(settings.get("direction_jitter_low_expansion_steps", 2))
DIRECTION_JITTER_SCORE_LOWER_PERCENTILE = float(settings.get("direction_jitter_score_lower_percentile", 0.0))
DIRECTION_JITTER_SCORE_UPPER_PERCENTILE = float(settings.get("direction_jitter_score_upper_percentile", 95.0))
score_model_artifacts = artifacts.get("score_models")
if not score_model_artifacts and artifacts:
    legacy_feature_names = np.asarray(artifacts["feature_names"])
    score_model_artifacts = {
        "combined": {
            "model_name": "combined",
            "model": artifacts["model"],
            "preprocess_pipeline": artifacts["preprocess_pipeline"],
            "feature_names": legacy_feature_names,
            "feature_weight_vector": np.asarray(artifacts.get("feature_weight_vector", np.ones(len(legacy_feature_names))), dtype=float),
            "score_calibration": {"lower": 0.0, "upper": 1.0},
            "score_column": "anomaly_score",
            "raw_score_column": "anomaly_score_raw",
            "expanded_feature_group_map": artifacts.get("expanded_feature_group_map", {}),
        }
    }
if score_model_artifacts is None:
    score_model_artifacts = {}

score_model_configs = artifacts.get("score_model_configs", {})
sync_score_config = artifacts.get("sync_score_config", {
    "audio_score_column": "audio_anomaly_score",
    "motion_score_column": "motion_anomaly_score",
    "max_lag_windows": 2,
})
final_score_weights = artifacts.get("final_score_weights", {
    "audio_anomaly_score": 0.45,
    "motion_anomaly_score": 0.35,
    "sync_score": 0.20,
})

default_flow_xy_impact_config = {
    "grid": settings.get("flow_grid", (3, 3)),
    "min_valid_cell_ratio": settings.get("flow_min_valid_cell_ratio", 0.01),
    "score_lower_quantile": settings.get("flow_score_lower_quantile", 0.50),
    "score_upper_quantile": settings.get("flow_score_upper_quantile", 0.99),
    "cell_score_upper_quantile": settings.get("flow_cell_score_upper_quantile", 0.95),
    "event_threshold": settings.get("flow_event_threshold", 0.65),
    "duration_alpha": settings.get("flow_duration_alpha", 1.0),
    "globality_weight": settings.get("flow_globality_weight", 0.5),
    "transience_weight": settings.get("flow_transience_weight", 0.3),
    "change_amount_weight": settings.get("flow_change_amount_weight", 0.2),
    "globality_ratio_weight": 0.85,
    "globality_intensity_weight": 0.15,
}
flow_xy_impact_config = artifacts.get("flow_xy_impact_config", default_flow_xy_impact_config)
flow_xy_impact_calibration = artifacts.get("flow_xy_impact_calibration", {})
FLOW_XY_IMPACT_COLUMNS = artifacts.get("flow_xy_impact_columns", [
    "front_flow_change_amount_score",
    "front_flow_globality_score",
    "front_flow_transience_score",
    "front_flow_xy_impact_score",
    "rear_flow_change_amount_score",
    "rear_flow_globality_score",
    "rear_flow_transience_score",
    "rear_flow_xy_impact_score",
])

# グラフ表示列は推論ノートブック側で制御します。
# 保存済みモデルにも plot_*_columns は入っていますが、検証中に手元でオン/オフしやすいようここを優先します。
PLOT_SCORE_COLUMNS = [
    "anomaly_score",
    "audio_anomaly_score",
    "motion_anomaly_score",
    "sync_score",
]
PLOT_FEATURE_COLUMNS = [
    "audio_rms",
    "audio_high_freq_energy",
    "front_broad_vibration_score",
    "rear_broad_vibration_score",
    "front_global_translation_norm",
    "rear_global_translation_norm",
    "front_flow_mag_mean",
    "rear_flow_mag_mean",
    "front_flow_x_mean",
    "rear_flow_x_mean",
    "front_flow_y_mean",
    "rear_flow_y_mean",
    "front_flow_change_amount_score",
    "rear_flow_change_amount_score",
    "front_flow_globality_score",
    "rear_flow_globality_score",
    "front_flow_transience_score",
    "rear_flow_transience_score",
    "front_flow_xy_impact_score",
    "rear_flow_xy_impact_score",
    "front_flow_high_change_cell_ratio",
    "rear_flow_high_change_cell_ratio",
    "front_flow_event_duration_steps",
    "rear_flow_event_duration_steps",
]

FPS_SAMPLE = settings.get("fps_sample", 10)
WINDOW_SEC = settings.get("window_sec", 0.2)
AUDIO_SR = settings.get("audio_sr", 16000)
N_MELS = settings.get("n_mels", 16)
FRAME_RESIZE_WIDTH = settings.get("frame_resize_width", 480)
FLOW_ANALYSIS_SCALE = float(settings.get("flow_analysis_scale", 0.75))
FLOW_GAUSSIAN_BLUR_KERNEL = settings.get("flow_gaussian_blur_kernel", (5, 5))
FLOW_GAUSSIAN_BLUR_SIGMA = float(settings.get("flow_gaussian_blur_sigma", 0.0))
FLOW_GRID = tuple(flow_xy_impact_config.get("grid", settings.get("flow_grid", (3, 3))))
FLOW_MIN_VALID_CELL_RATIO = float(flow_xy_impact_config.get("min_valid_cell_ratio", 0.01))
FLOW_SCORE_LOWER_QUANTILE = float(flow_xy_impact_config.get("score_lower_quantile", 0.50))
FLOW_SCORE_UPPER_QUANTILE = float(flow_xy_impact_config.get("score_upper_quantile", 0.99))
FLOW_CELL_SCORE_UPPER_QUANTILE = float(flow_xy_impact_config.get("cell_score_upper_quantile", 0.95))
FLOW_EVENT_THRESHOLD = float(flow_xy_impact_config.get("event_threshold", 0.65))
FLOW_DURATION_ALPHA = float(flow_xy_impact_config.get("duration_alpha", 1.0))
FLOW_GLOBALITY_WEIGHT = float(flow_xy_impact_config.get("globality_weight", 0.5))
FLOW_TRANSIENCE_WEIGHT = float(flow_xy_impact_config.get("transience_weight", 0.3))
FLOW_CHANGE_AMOUNT_WEIGHT = float(flow_xy_impact_config.get("change_amount_weight", 0.2))
FLOW_GLOBALITY_RATIO_WEIGHT = float(flow_xy_impact_config.get("globality_ratio_weight", 0.85))
FLOW_GLOBALITY_INTENSITY_WEIGHT = float(flow_xy_impact_config.get("globality_intensity_weight", 0.15))
MAX_CORNERS = 200
TOP_N_ANOMALIES = 8
BROAD_VIBRATION_TOP_K = int(settings.get("broad_vibration_top_k", artifacts.get("broad_vibration_top_k", 5)))
BROAD_VIBRATION_COLUMNS = list(artifacts.get("broad_vibration_columns", settings.get("broad_vibration_columns", [
    "front_broad_vibration_score",
    "rear_broad_vibration_score",
])))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"MODEL_PATH  : {MODEL_PATH}")
print(f"model_version: {artifacts.get('model_version', 'legacy')}")
print(f"settings    : {settings}")
print(f"RUN_INFERENCE: {RUN_INFERENCE}")
print(f"PLOT_EXISTING_GRAPHS: {PLOT_EXISTING_GRAPHS}")
print(f"PLOT_RAW_FLOW_XY: {PLOT_RAW_FLOW_XY}")
print(f"RAW_FLOW_MAGNITUDE_LOWPASS_CUTOFF_HZ: {RAW_FLOW_MAGNITUDE_LOWPASS_CUTOFF_HZ}")
print(f"RAW_FLOW_MAGNITUDE_LOWPASS_ORDER: {RAW_FLOW_MAGNITUDE_LOWPASS_ORDER}")
print(f"FLOW_SCORE_WINDOW_SEC: {FLOW_SCORE_WINDOW_SEC}")
print(f"FLOW_GAUSSIAN_BLUR_KERNEL: {FLOW_GAUSSIAN_BLUR_KERNEL}")
print(f"FLOW_GAUSSIAN_BLUR_SIGMA : {FLOW_GAUSSIAN_BLUR_SIGMA}")
print(f"FLOW_SCORE_HOP_SEC: {FLOW_SCORE_HOP_SEC}")
print(f"DIRECTION_JITTER_HIGH_THRESHOLD: {DIRECTION_JITTER_HIGH_THRESHOLD}")
print(f"DIRECTION_JITTER_LOW_THRESHOLD : {DIRECTION_JITTER_LOW_THRESHOLD}")
print(f"DIRECTION_JITTER_THRESHOLD_MODE : {DIRECTION_JITTER_THRESHOLD_MODE}")
print(f"DIRECTION_JITTER_HIGH_PERCENTILE: {DIRECTION_JITTER_HIGH_PERCENTILE}")
print(f"DIRECTION_JITTER_LOW_PERCENTILE : {DIRECTION_JITTER_LOW_PERCENTILE}")
print(f"DIRECTION_JITTER_LOW_EXPANSION_STEPS: {DIRECTION_JITTER_LOW_EXPANSION_STEPS}")
print(f"DIRECTION_JITTER_SCORE_LOWER_PERCENTILE: {DIRECTION_JITTER_SCORE_LOWER_PERCENTILE}")
print(f"DIRECTION_JITTER_SCORE_UPPER_PERCENTILE: {DIRECTION_JITTER_SCORE_UPPER_PERCENTILE}")
print(f"FLOW_SCORE_DISTINCTIVENESS_WEIGHT: {FLOW_SCORE_DISTINCTIVENESS_WEIGHT}")
print(f"BROAD_VIBRATION_TOP_K: {BROAD_VIBRATION_TOP_K}")
print(f"BROAD_VIBRATION_COLUMNS: {BROAD_VIBRATION_COLUMNS}")
print(f"score models: {list(score_model_artifacts.keys())}")
for model_name, artifact in score_model_artifacts.items():
    print(f"- {model_name}: feature_dim={len(artifact['feature_names'])}, score_column={artifact.get('score_column')}")
print(f"flow xy impact config : {flow_xy_impact_config}")
print(f"flow xy calibration prefixes: {list(flow_xy_impact_calibration.get('prefix_stats', {}).keys()) if isinstance(flow_xy_impact_calibration, dict) else []}")
print(f"sync score config : {sync_score_config}")
print(f"final score weights: {final_score_weights}")
print(f"plot score columns : {PLOT_SCORE_COLUMNS}")
print(f"plot feature columns: {PLOT_FEATURE_COLUMNS}")


PROJECT_ROOT: /workspaces/forklift
MODEL_PATH  : /workspaces/forklift/outputs/anomaly_feature_poc/isolation_forest_feature_poc.joblib
model_version: audio_motion_sync_v9_broad_vibration
settings    : {'fps_sample': 10, 'window_sec': 0.2, 'audio_sr': 16000, 'n_mels': 16, 'frame_resize_width': 480, 'flow_analysis_scale': 0.75, 'flow_grid': (3, 3), 'motion_feature': 'broad_vibration', 'flow_change_feature': 'xy_impact', 'flow_min_valid_cell_ratio': 0.01, 'flow_score_lower_quantile': 0.5, 'flow_score_upper_quantile': 0.99, 'flow_cell_score_upper_quantile': 0.95, 'flow_event_threshold': 0.65, 'flow_duration_alpha': 1.0, 'flow_globality_weight': 0.5, 'flow_transience_weight': 0.3, 'flow_change_amount_weight': 0.2, 'flow_score_window_sec': 1.0, 'flow_score_hop_sec': 0.5, 'flow_score_alpha_min': 0.04, 'flow_score_alpha_max': 0.42, 'flow_score_min_visible': 1e-06, 'flow_score_high_ratio_fraction': 0.5, 'flow_direction_min_mag': 0.025, 'direction_jitter_high_threshold': 1.5, 'direction_jitter_lo

## 2. 推論対象

`TARGET_SPECS` に推論したい前処理済み動画を追加します。`sample_id` は `*_front.mp4` / `*_rear.mp4` / `*_audio.wav` の共通プレフィックスです。

In [ ]:
TARGET_SPECS = [
    {"sample_id": "001_後進時にトラックに衝突", "category": "anomaly", "environment": "outdoor"},
    {"sample_id": "005_後進旋回時トラックに衝突", "category": "anomaly", "environment": "outdoor"},
    {"sample_id": "006_安全ポールに接触", "category": "anomaly", "environment": "indoor"},
    {"sample_id": "012_安全ポール乗り上げ", "category": "anomaly", "environment": "indoor"},
    {"sample_id": "090_前進時、電源ケーブル乗り上げ", "category": "anomaly", "environment": "indoor"},
    {"sample_id": "1001", "category": "normal", "environment": "outdoor"},
    {"sample_id": "1002", "category": "normal", "environment": "outdoor"},
    {"sample_id": "1006", "category": "normal", "environment": "outdoor"},
    {"sample_id": "1024", "category": "normal", "environment": "indoor"},
    {"sample_id": "1036", "category": "normal", "environment": "indoor"},
    # {"sample_id": "1038", "category": "normal", "environment": "indoor"},
    # {"sample_id": "1065", "category": "normal", "environment": "indoor"},
    # {"sample_id": "1066", "category": "normal", "environment": "indoor"},
    # {"sample_id": "1067", "category": "normal", "environment": "indoor"},
    # {"sample_id": "1068", "category": "normal", "environment": "indoor"},
    # {"sample_id": "1069", "category": "normal", "environment": "indoor"},
    # {"sample_id": "1070", "category": "normal", "environment": "indoor"},
]

TARGET_OUTPUT_ROOT = OUTPUT_DIR / "target_inference"
TARGET_COMPARISON_DIR = TARGET_OUTPUT_ROOT / "_comparison"
TARGET_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TARGET_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)


def build_target_sample(spec: dict) -> pd.Series:
    sample_id = spec["sample_id"]
    category = spec.get("category", "anomaly")
    environment = spec.get("environment", "outdoor")
    target_dir = PROJECT_ROOT / "data" / "movie_preprocess" / category / environment
    output_dir = TARGET_OUTPUT_ROOT / sample_id
    output_dir.mkdir(parents=True, exist_ok=True)
    short_id = sample_id.split("_", 1)[0]
    return pd.Series({
        "sample_id": sample_id,
        "category": category,
        "environment": environment,
        "plot_label": spec.get("label", f"{category}/{environment}/{short_id}"),
        "front_path": target_dir / f"{sample_id}_front.mp4",
        "rear_path": target_dir / f"{sample_id}_rear.mp4",
        "audio_path": target_dir / f"{sample_id}_audio.wav",
        "output_dir": output_dir,
    })


if not TARGET_SPECS:
    raise ValueError("TARGET_SPECS must contain at least one target.")

TARGET_SAMPLES = [build_target_sample(spec) for spec in TARGET_SPECS]
for sample in TARGET_SAMPLES:
    missing = [str(sample[c]) for c in ["front_path", "rear_path", "audio_path"] if not Path(sample[c]).exists()]
    if missing:
        raise FileNotFoundError(f"Missing files for {sample['sample_id']}:\n" + "\n".join(missing))

print(f"targets: {len(TARGET_SAMPLES)}")
for sample in TARGET_SAMPLES:
    print(f"- {sample['plot_label']}: {sample['sample_id']}")

targets: 4
- anomaly/outdoor/001: 001_後進時にトラックに衝突
- anomaly/outdoor/005: 005_後進旋回時トラックに衝突
- normal/outdoor/1001: 1001
- normal/indoor/1036: 1036


## 3. 特徴量抽出

In [106]:
def resize_keep_aspect(frame: np.ndarray, width: int | None) -> np.ndarray:
    if width is None or frame.shape[1] <= width:
        return frame
    scale = width / frame.shape[1]
    height = max(1, int(round(frame.shape[0] * scale)))
    return cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)


def extract_video_frames(video_path: str | Path, fps_sample: float = FPS_SAMPLE, resize_width: int | None = FRAME_RESIZE_WIDTH) -> list[dict]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        warnings.warn(f"Could not open video: {video_path}")
        return []
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    if not src_fps or np.isnan(src_fps) or src_fps <= 0:
        src_fps = fps_sample
    interval = max(1, int(round(src_fps / fps_sample)))

    frames = []
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % interval == 0:
            frames.append({"time": frame_idx / src_fps, "frame": resize_keep_aspect(frame, resize_width)})
        frame_idx += 1
    cap.release()
    return frames


def load_audio_mono(wav_path: str | Path, sr: int = AUDIO_SR) -> tuple[np.ndarray, int]:
    if librosa is not None:
        y, used_sr = librosa.load(wav_path, sr=sr, mono=True)
        return y.astype(np.float32), used_sr
    used_sr, y = wavfile.read(wav_path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    max_abs = np.max(np.abs(y)) if y.size else 1.0
    if max_abs > 0:
        y = y / max_abs
    if used_sr != sr and y.size:
        gcd = math.gcd(used_sr, sr)
        y = resample_poly(y, sr // gcd, used_sr // gcd).astype(np.float32)
        used_sr = sr
    return y, used_sr


def extract_audio_features(wav_path: str | Path, video_id: str) -> pd.DataFrame:
    y, used_sr = load_audio_mono(wav_path, sr=AUDIO_SR)
    win = max(1, int(round(WINDOW_SEC * used_sr)))
    if y.size == 0:
        return pd.DataFrame(columns=["video_id", "time_bin", "time", "audio_missing"])

    rows = []
    for i in range(int(math.ceil(len(y) / win))):
        segment = y[i * win : min(len(y), (i + 1) * win)]
        if segment.size == 0:
            continue
        spectrum = np.abs(np.fft.rfft(segment * np.hanning(segment.size)))
        freqs = np.fft.rfftfreq(segment.size, d=1.0 / used_sr)
        power = spectrum ** 2
        power_sum = float(power.sum()) + 1e-12
        centroid = float((freqs * power).sum() / power_sum)
        rows.append({
            "video_id": video_id,
            "time_bin": i,
            "time": i * WINDOW_SEC,
            "audio_rms": float(np.sqrt(np.mean(segment ** 2))),
            "audio_energy": float(np.mean(segment ** 2)),
            "audio_peak": float(np.max(np.abs(segment))),
            "audio_ptp": float(np.ptp(segment)),
            "audio_zcr": float(np.mean(np.abs(np.diff(np.signbit(segment))).astype(float))) if segment.size > 1 else 0.0,
            "audio_centroid": centroid,
            "audio_bandwidth": float(np.sqrt((((freqs - centroid) ** 2) * power).sum() / power_sum)),
            "audio_high_freq_energy": float(power[freqs >= 3000].sum() / power_sum),
            "audio_missing": 0,
        })
    df = pd.DataFrame(rows)

    if librosa is not None and len(df):
        mel = librosa.feature.melspectrogram(
            y=y, sr=used_sr, n_mels=N_MELS, n_fft=min(2048, max(256, win)), hop_length=win, power=2.0
        )
        log_mel = librosa.power_to_db(mel, ref=np.max).T
        for m in range(N_MELS):
            values = np.full(len(df), np.nan, dtype=float)
            values[: min(len(values), log_mel.shape[0])] = log_mel[: min(len(values), log_mel.shape[0]), m]
            df[f"audio_mel_{m:02d}"] = values
    else:
        for m in range(N_MELS):
            df[f"audio_mel_{m:02d}"] = 0.0
    return df

In [107]:
def build_motion_valid_mask(frame_shape: tuple[int, ...], prefix: str | None = None) -> np.ndarray:
    """Return an all-true mask. Region masking is intentionally disabled."""
    h, w = int(frame_shape[0]), int(frame_shape[1])
    return np.ones((h, w), dtype=bool)


def build_feature_track_mask(frame_shape: tuple[int, ...], prefix: str | None = None) -> np.ndarray:
    valid = build_motion_valid_mask(frame_shape, prefix)
    return (valid.astype(np.uint8) * 255)


def masked_values(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> np.ndarray:
    selected = values[valid_mask]
    if selected.size == 0 and fallback_to_all:
        selected = values.reshape(-1)
    return selected[np.isfinite(selected)]


def masked_mean(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.mean(selected)) if selected.size else 0.0


def masked_std(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.std(selected)) if selected.size else 0.0


def masked_max(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.max(selected)) if selected.size else 0.0


def points_in_valid_mask(points: np.ndarray, valid_mask: np.ndarray) -> np.ndarray:
    h, w = valid_mask.shape
    x = np.rint(points[:, 0]).astype(int)
    y = np.rint(points[:, 1]).astype(int)
    in_bounds = (0 <= x) & (x < w) & (0 <= y) & (y < h)
    keep = np.zeros(points.shape[0], dtype=bool)
    keep[in_bounds] = valid_mask[y[in_bounds], x[in_bounds]]
    return keep


def normalize_gaussian_blur_kernel(kernel: tuple[int, int] | list[int] | int | float) -> tuple[int, int]:
    if isinstance(kernel, (int, float)):
        kx = ky = int(kernel)
    else:
        values = list(kernel)
        if not values:
            return 0, 0
        kx = int(values[0])
        ky = int(values[1]) if len(values) > 1 else kx
    if kx <= 1 or ky <= 1:
        return 0, 0
    return kx + (kx % 2 == 0), ky + (ky % 2 == 0)


def apply_flow_gaussian_blur(gray: np.ndarray) -> np.ndarray:
    kx, ky = normalize_gaussian_blur_kernel(FLOW_GAUSSIAN_BLUR_KERNEL)
    if kx <= 1 or ky <= 1:
        return gray
    return cv2.GaussianBlur(gray, (kx, ky), sigmaX=FLOW_GAUSSIAN_BLUR_SIGMA, sigmaY=FLOW_GAUSSIAN_BLUR_SIGMA)


def resize_gray_for_flow(gray: np.ndarray, scale: float = FLOW_ANALYSIS_SCALE) -> tuple[np.ndarray, float, float]:
    """Downscale and blur the image used by Farneback flow, then return x/y scale factors."""
    scale = float(scale)
    if scale <= 0.0 or scale >= 1.0:
        return apply_flow_gaussian_blur(gray), 1.0, 1.0
    h, w = gray.shape[:2]
    new_w = max(8, int(round(w * scale)))
    new_h = max(8, int(round(h * scale)))
    if new_w == w and new_h == h:
        return apply_flow_gaussian_blur(gray), 1.0, 1.0
    resized = cv2.resize(gray, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return apply_flow_gaussian_blur(resized), new_w / max(w, 1), new_h / max(h, 1)

def flow_xy_raw_output_columns(prefix: str, grid: tuple[int, int] = FLOW_GRID) -> list[str]:
    cols = [
        f"{prefix}_flow_change_amount_raw",
        f"{prefix}_flow_cell_change_mean_raw",
        f"{prefix}_flow_cell_change_max_raw",
        f"{prefix}_flow_cell_change_p95_raw",
        f"{prefix}_flow_valid_cell_count",
        f"{prefix}_flow_cell_count",
    ]
    for gy in range(grid[0]):
        for gx in range(grid[1]):
            cols.append(f"{prefix}_flow_cell_{gy}_{gx}_change_raw")
    return cols


def _empty_flow_row(prefix: str, time_sec: float, time_bin: int, video_id: str | None, flow_dt_sec: float | None = None) -> dict:
    default_dt = 1.0 / max(float(FPS_SAMPLE), 1e-6)
    row = {
        "video_id": video_id,
        "time_bin": time_bin,
        "time": time_sec,
        f"{prefix}_flow_dt_sec": float(flow_dt_sec if flow_dt_sec is not None else default_dt),
    }
    base_cols = [
        "flow_mag_mean", "flow_mag_std", "flow_mag_max", "flow_angle_mean", "flow_angle_std",
        "flow_x_mean", "flow_x_std", "flow_y_mean", "flow_y_std", "flow_failed",
    ]
    for c in base_cols:
        row[f"{prefix}_{c}"] = 1.0 if c == "flow_failed" else 0.0
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            for c in ["mag_mean", "mag_std", "x_mean", "y_mean", "valid_ratio"]:
                row[f"{prefix}_flow_cell_{gy}_{gx}_{c}"] = 0.0
    for col in flow_xy_raw_output_columns(prefix):
        row[col] = 0.0
    return row


def _diff_per_second(work: pd.DataFrame, col: str, dt: pd.Series) -> np.ndarray:
    delta = work.groupby("video_id", dropna=False)[col].diff().fillna(0.0).to_numpy(dtype=float)
    return delta / dt.to_numpy(dtype=float)


def _nanmean_or_zero(values: np.ndarray, axis: int = 1) -> np.ndarray:
    with np.errstate(invalid="ignore"):
        out = np.nanmean(values, axis=axis)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def _nanmax_or_zero(values: np.ndarray, axis: int = 1) -> np.ndarray:
    all_nan = np.isnan(values).all(axis=axis)
    safe = np.where(np.isnan(values), -np.inf, values)
    out = np.max(safe, axis=axis)
    out[all_nan] = 0.0
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def _nanpercentile_or_zero(values: np.ndarray, percentile: float, axis: int = 1) -> np.ndarray:
    out = np.zeros(values.shape[0], dtype=float)
    for i, row in enumerate(values):
        finite = row[np.isfinite(row)]
        if finite.size:
            out[i] = float(np.percentile(finite, percentile))
    return out


def add_flow_xy_change_window_features(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Add raw x/y flow-difference features. Normal-data calibration is applied later."""
    if df.empty:
        return df
    x_col = f"{prefix}_flow_x_mean"
    y_col = f"{prefix}_flow_y_mean"
    if x_col not in df.columns or y_col not in df.columns:
        return df.drop(columns=[f"{prefix}_flow_dt_sec"], errors="ignore")

    work = df.sort_values(["video_id", "time"]).reset_index(drop=True).copy()
    default_dt = 1.0 / max(float(FPS_SAMPLE), 1e-6)
    dt_col = f"{prefix}_flow_dt_sec"
    if dt_col in work.columns:
        dt = work[dt_col].replace([np.inf, -np.inf], np.nan).fillna(default_dt).clip(lower=1e-6)
    else:
        dt = work.groupby("video_id", dropna=False)["time"].diff().fillna(default_dt).clip(lower=1e-6)

    dx = _diff_per_second(work, x_col, dt)
    dy = _diff_per_second(work, y_col, dt)
    amount_col = f"{prefix}_flow_change_amount_raw"
    mean_col = f"{prefix}_flow_cell_change_mean_raw"
    max_col = f"{prefix}_flow_cell_change_max_raw"
    p95_col = f"{prefix}_flow_cell_change_p95_raw"
    valid_count_col = f"{prefix}_flow_valid_cell_count"
    cell_count_col = f"{prefix}_flow_cell_count"
    work[amount_col] = np.hypot(dx, dy)

    cell_change_cols = []
    cell_changes = []
    valid_masks = []
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            cx = f"{prefix}_flow_cell_{gy}_{gx}_x_mean"
            cy = f"{prefix}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{prefix}_flow_cell_{gy}_{gx}_valid_ratio"
            change_col = f"{prefix}_flow_cell_{gy}_{gx}_change_raw"
            if cx not in work.columns or cy not in work.columns:
                continue
            cdx = _diff_per_second(work, cx, dt)
            cdy = _diff_per_second(work, cy, dt)
            change = np.hypot(cdx, cdy)
            if valid_col in work.columns:
                valid = work[valid_col].to_numpy(dtype=float) >= float(FLOW_MIN_VALID_CELL_RATIO)
            else:
                valid = np.ones(len(work), dtype=bool)
            change = np.where(valid, change, np.nan)
            work[change_col] = change
            cell_change_cols.append(change_col)
            cell_changes.append(change)
            valid_masks.append(valid)

    if cell_changes:
        change_matrix = np.vstack(cell_changes).T
        valid_matrix = np.vstack(valid_masks).T
        work[mean_col] = _nanmean_or_zero(change_matrix, axis=1)
        work[max_col] = _nanmax_or_zero(change_matrix, axis=1)
        work[p95_col] = _nanpercentile_or_zero(change_matrix, 95, axis=1)
        work[valid_count_col] = valid_matrix.sum(axis=1).astype(float)
        work[cell_count_col] = float(len(cell_change_cols))
    else:
        work[mean_col] = 0.0
        work[max_col] = 0.0
        work[p95_col] = 0.0
        work[valid_count_col] = 0.0
        work[cell_count_col] = 0.0

    agg_spec = {
        amount_col: (amount_col, "max"),
        mean_col: (mean_col, "max"),
        max_col: (max_col, "max"),
        p95_col: (p95_col, "max"),
        valid_count_col: (valid_count_col, "max"),
        cell_count_col: (cell_count_col, "max"),
    }
    for col in cell_change_cols:
        agg_spec[col] = (col, "max")

    window_features = work.groupby(["video_id", "time_bin"], as_index=False).agg(**agg_spec)
    out = df.merge(window_features, on=["video_id", "time_bin"], how="left")
    for col in flow_xy_raw_output_columns(prefix):
        if col not in out.columns:
            out[col] = 0.0
        else:
            out[col] = out[col].fillna(0.0)
    return out.drop(columns=[f"{prefix}_flow_dt_sec"], errors="ignore")


def compute_optical_flow_features(
    frames: list[dict],
    prefix: str,
    window_sec: float = WINDOW_SEC,
    grid: tuple[int, int] = FLOW_GRID,
    video_id: str | None = None,
) -> pd.DataFrame:
    if len(frames) < 2:
        return pd.DataFrame()

    rows = []
    prev_gray_original = cv2.cvtColor(frames[0]["frame"], cv2.COLOR_BGR2GRAY)
    prev_gray, prev_scale_x, prev_scale_y = resize_gray_for_flow(prev_gray_original)
    prev_time = float(frames[0]["time"])
    for item in frames[1:]:
        time_sec = float(item["time"])
        flow_dt_sec = max(time_sec - prev_time, 1e-6)
        time_bin = int(round(time_sec / window_sec))
        gray_original = cv2.cvtColor(item["frame"], cv2.COLOR_BGR2GRAY)
        gray, scale_x, scale_y = resize_gray_for_flow(gray_original)
        try:
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None,
                pyr_scale=0.5, levels=3, winsize=15, iterations=3,
                poly_n=5, poly_sigma=1.2, flags=0,
            )
            # Farneback returns displacement in the downscaled image coordinate.
            # Convert it back to the resized-frame coordinate so thresholds stay comparable.
            fx = flow[..., 0] / max(scale_x, 1e-6)
            fy = flow[..., 1] / max(scale_y, 1e-6)
            mag, angle = cv2.cartToPolar(fx, fy, angleInDegrees=False)
            valid_mask = build_motion_valid_mask(mag.shape, prefix)
            row = {
                "video_id": video_id,
                "time_bin": time_bin,
                "time": time_sec,
                f"{prefix}_flow_dt_sec": flow_dt_sec,
                f"{prefix}_flow_mag_mean": masked_mean(mag, valid_mask),
                f"{prefix}_flow_mag_std": masked_std(mag, valid_mask),
                f"{prefix}_flow_mag_max": masked_max(mag, valid_mask),
                f"{prefix}_flow_angle_mean": masked_mean(angle, valid_mask),
                f"{prefix}_flow_angle_std": masked_std(angle, valid_mask),
                f"{prefix}_flow_x_mean": masked_mean(fx, valid_mask),
                f"{prefix}_flow_x_std": masked_std(fx, valid_mask),
                f"{prefix}_flow_y_mean": masked_mean(fy, valid_mask),
                f"{prefix}_flow_y_std": masked_std(fy, valid_mask),
                f"{prefix}_flow_failed": 0,
            }
            h, w = mag.shape
            gy_n, gx_n = grid
            for gy in range(gy_n):
                y0, y1 = int(h * gy / gy_n), int(h * (gy + 1) / gy_n)
                for gx in range(gx_n):
                    x0, x1 = int(w * gx / gx_n), int(w * (gx + 1) / gx_n)
                    cell_mag = mag[y0:y1, x0:x1]
                    cell_fx = fx[y0:y1, x0:x1]
                    cell_fy = fy[y0:y1, x0:x1]
                    cell_mask = valid_mask[y0:y1, x0:x1]
                    row[f"{prefix}_flow_cell_{gy}_{gx}_mag_mean"] = masked_mean(cell_mag, cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_mag_std"] = masked_std(cell_mag, cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_x_mean"] = masked_mean(cell_fx, cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_y_mean"] = masked_mean(cell_fy, cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_valid_ratio"] = float(np.mean(cell_mask)) if cell_mask.size else 0.0
        except Exception as exc:
            warnings.warn(f"{prefix} optical flow failed at {time_sec:.2f}s: {exc}")
            row = _empty_flow_row(prefix, time_sec, time_bin, video_id, flow_dt_sec=flow_dt_sec)
        rows.append(row)
        prev_gray = gray
        prev_scale_x = scale_x
        prev_scale_y = scale_y
        prev_time = time_sec

    return add_flow_xy_change_window_features(pd.DataFrame(rows), prefix)



def _global_failure_row(prefix: str, time_sec: float, time_bin: int, video_id: str, num_points: int = 0) -> dict:
    return {
        "video_id": video_id, "time_bin": time_bin, "time": time_sec,
        f"{prefix}_global_dx": 0.0, f"{prefix}_global_dy": 0.0,
        f"{prefix}_global_translation_norm": 0.0, f"{prefix}_global_rotation": 0.0,
        f"{prefix}_global_scale": 1.0, f"{prefix}_global_ransac_residual": 0.0,
        f"{prefix}_global_inlier_ratio": 0.0,
        f"{prefix}_global_num_tracked_points": float(num_points),
        f"{prefix}_global_failed": 1,
    }


def compute_global_motion_features(frames: list[dict], prefix: str, video_id: str) -> pd.DataFrame:
    if len(frames) < 2:
        return pd.DataFrame()
    lk_params = dict(winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    feature_params = dict(maxCorners=MAX_CORNERS, qualityLevel=0.01, minDistance=7, blockSize=7)

    def detect_motion_points(gray_image: np.ndarray) -> np.ndarray | None:
        return cv2.goodFeaturesToTrack(gray_image, mask=build_feature_track_mask(gray_image.shape, prefix), **feature_params)

    rows = []
    prev_gray = cv2.cvtColor(frames[0]["frame"], cv2.COLOR_BGR2GRAY)
    prev_valid_mask = build_motion_valid_mask(prev_gray.shape, prefix)
    prev_pts = detect_motion_points(prev_gray)

    for item in frames[1:]:
        time_sec = float(item["time"])
        time_bin = int(round(time_sec / WINDOW_SEC))
        gray = cv2.cvtColor(item["frame"], cv2.COLOR_BGR2GRAY)
        current_valid_mask = build_motion_valid_mask(gray.shape, prefix)
        if prev_pts is None or len(prev_pts) < 6:
            rows.append(_global_failure_row(prefix, time_sec, time_bin, video_id, 0 if prev_pts is None else len(prev_pts)))
            prev_gray, prev_valid_mask, prev_pts = gray, current_valid_mask, detect_motion_points(gray)
            continue
        next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, **lk_params)
        if next_pts is None or status is None:
            rows.append(_global_failure_row(prefix, time_sec, time_bin, video_id, len(prev_pts)))
            prev_gray, prev_valid_mask, prev_pts = gray, current_valid_mask, detect_motion_points(gray)
            continue
        good_prev = prev_pts[status.ravel() == 1].reshape(-1, 2)
        good_next = next_pts[status.ravel() == 1].reshape(-1, 2)
        track_mask = points_in_valid_mask(good_prev, prev_valid_mask) & points_in_valid_mask(good_next, current_valid_mask)
        good_prev = good_prev[track_mask]
        good_next = good_next[track_mask]
        if len(good_prev) < 6:
            rows.append(_global_failure_row(prefix, time_sec, time_bin, video_id, len(good_prev)))
            prev_gray, prev_valid_mask, prev_pts = gray, current_valid_mask, detect_motion_points(gray)
            continue
        M, inliers = cv2.estimateAffinePartial2D(good_prev, good_next, method=cv2.RANSAC, ransacReprojThreshold=3.0)
        if M is None or inliers is None:
            rows.append(_global_failure_row(prefix, time_sec, time_bin, video_id, len(good_prev)))
            prev_gray, prev_valid_mask, prev_pts = gray, current_valid_mask, detect_motion_points(gray)
            continue
        a, _, dx = M[0]
        c, _, dy = M[1]
        inlier_mask = inliers.ravel().astype(bool)
        pred = cv2.transform(good_prev.reshape(-1, 1, 2), M).reshape(-1, 2)
        residuals = np.linalg.norm(pred - good_next, axis=1)
        rows.append({
            "video_id": video_id, "time_bin": time_bin, "time": time_sec,
            f"{prefix}_global_dx": float(dx),
            f"{prefix}_global_dy": float(dy),
            f"{prefix}_global_translation_norm": float(np.hypot(dx, dy)),
            f"{prefix}_global_rotation": float(np.arctan2(c, a)),
            f"{prefix}_global_scale": float(np.sqrt(a * a + c * c)),
            f"{prefix}_global_ransac_residual": float(np.mean(residuals[inlier_mask])) if inlier_mask.any() else float(np.mean(residuals)),
            f"{prefix}_global_inlier_ratio": float(inlier_mask.mean()),
            f"{prefix}_global_num_tracked_points": float(len(good_prev)),
            f"{prefix}_global_failed": 0,
        })
        prev_gray, prev_valid_mask, prev_pts = gray, current_valid_mask, detect_motion_points(gray)
    return pd.DataFrame(rows)

In [108]:
def aggregate_feature_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    if "time_bin" not in df.columns:
        df["time_bin"] = np.round(df["time"] / WINDOW_SEC).astype(int)
    key_cols = ["video_id", "time_bin"]
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in {"time", "time_bin"}]
    other_cols = [c for c in df.columns if c not in set(key_cols + numeric_cols + ["time"])]
    out = df.groupby(key_cols, as_index=False)[numeric_cols].mean() if numeric_cols else df[key_cols].drop_duplicates()
    for col in other_cols:
        values = df.groupby(key_cols)[col].agg(lambda x: x.dropna().mode().iat[0] if len(x.dropna().mode()) else "unknown").reset_index()
        out = out.merge(values, on=key_cols, how="left")
    out["time"] = out["time_bin"] * WINDOW_SEC
    return out


def align_features_by_time(feature_dfs: list[pd.DataFrame]) -> pd.DataFrame:
    valid = [aggregate_feature_df(df) for df in feature_dfs if df is not None and len(df)]
    if not valid:
        return pd.DataFrame()
    merged = valid[0]
    for df in valid[1:]:
        merged = merged.merge(df.drop(columns=["time"], errors="ignore"), on=["video_id", "time_bin"], how="outer")
    merged["time"] = merged["time_bin"] * WINDOW_SEC
    merged = merged.sort_values(["video_id", "time_bin"]).reset_index(drop=True)
    numeric_cols = [c for c in merged.select_dtypes(include=[np.number]).columns if c not in {"time", "time_bin"}]
    merged[numeric_cols] = merged.groupby("video_id", group_keys=False)[numeric_cols].apply(lambda g: g.ffill().bfill()).fillna(0.0)
    for c in merged.select_dtypes(exclude=[np.number]).columns:
        if c != "video_id":
            merged[c] = merged[c].fillna("unknown")
    return merged


FLOW_XY_SCORE_SUFFIXES = (
    "flow_change_amount_score",
    "flow_globality_score",
    "flow_transience_score",
    "flow_xy_impact_score",
)
FLOW_XY_DIAGNOSTIC_SUFFIXES = (
    "flow_change_amount_raw",
    "flow_cell_change_mean_raw",
    "flow_cell_change_max_raw",
    "flow_cell_change_p95_raw",
    "flow_high_change_cell_count",
    "flow_valid_cell_count",
    "flow_high_change_cell_ratio",
    "flow_event_duration_steps",
    "flow_cell_count",
)


def flow_xy_score_columns(prefix: str) -> list[str]:
    return [f"{prefix}_{suffix}" for suffix in FLOW_XY_SCORE_SUFFIXES]


def flow_xy_diagnostic_columns(prefix: str) -> list[str]:
    return [f"{prefix}_{suffix}" for suffix in FLOW_XY_DIAGNOSTIC_SUFFIXES]


def add_flow_change_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    sort_cols = [c for c in ["video_id", "time_bin", "time"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)
    for prefix in ["front", "rear"]:
        for col in [*flow_xy_score_columns(prefix), *flow_xy_diagnostic_columns(prefix)]:
            if col not in df.columns:
                df[col] = 0.0
    return df


def add_cross_camera_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for name, front_col, rear_col in [
        ("flow_mag_mean", "front_flow_mag_mean", "rear_flow_mag_mean"),
        ("flow_xy_impact_score", "front_flow_xy_impact_score", "rear_flow_xy_impact_score"),
        ("flow_globality_score", "front_flow_globality_score", "rear_flow_globality_score"),
        ("flow_change_amount_score", "front_flow_change_amount_score", "rear_flow_change_amount_score"),
        ("global_translation_norm", "front_global_translation_norm", "rear_global_translation_norm"),
        ("global_rotation_abs", "front_global_rotation", "rear_global_rotation"),
    ]:
        if front_col in df.columns and rear_col in df.columns:
            f = df[front_col].abs() if "rotation" in name else df[front_col]
            r = df[rear_col].abs() if "rotation" in name else df[rear_col]
            df[f"front_rear_{name}_mean"] = (f + r) / 2
            df[f"front_rear_{name}_diff_abs"] = (f - r).abs()
            df[f"front_rear_{name}_both_high"] = f * r
    return df


def _score_from_scale(values, scale: dict[str, float]) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    lower = float(scale.get("lower", 0.0))
    upper = float(scale.get("upper", lower + 1.0))
    denom = max(upper - lower, 1e-6)
    return np.clip((np.nan_to_num(arr, nan=lower, posinf=upper, neginf=lower) - lower) / denom, 0.0, 1.0)


def _event_duration_steps(active: np.ndarray) -> np.ndarray:
    active = np.asarray(active, dtype=bool)
    durations = np.zeros(active.size, dtype=float)
    start = None
    for i, flag in enumerate(active):
        if flag and start is None:
            start = i
        ended = (not flag or i == active.size - 1) and start is not None
        if ended:
            end = i if not flag else i + 1
            durations[start:end] = float(end - start)
            start = None
    return durations


def _duration_by_video(df: pd.DataFrame, active: np.ndarray) -> pd.Series:
    durations = pd.Series(0.0, index=df.index, name="event_duration_steps")
    active_series = pd.Series(active, index=df.index)
    ordered = df.sort_values([c for c in ["video_id", "time_bin", "time"] if c in df.columns])
    for _, group in ordered.groupby("video_id", sort=False, dropna=False):
        durations.loc[group.index] = _event_duration_steps(active_series.loc[group.index].to_numpy(dtype=bool))
    return durations


def apply_flow_xy_impact_calibration(df: pd.DataFrame, calibration: dict) -> pd.DataFrame:
    result = add_flow_change_features(df)
    prefix_stats = calibration.get("prefix_stats", {}) if isinstance(calibration, dict) else {}
    weight_sum = max(FLOW_GLOBALITY_WEIGHT + FLOW_TRANSIENCE_WEIGHT + FLOW_CHANGE_AMOUNT_WEIGHT, 1e-6)
    for prefix in ["front", "rear"]:
        stats = prefix_stats.get(prefix, {})
        amount_col = f"{prefix}_flow_change_amount_raw"
        amount_score_col = f"{prefix}_flow_change_amount_score"
        globality_col = f"{prefix}_flow_globality_score"
        transience_col = f"{prefix}_flow_transience_score"
        impact_col = f"{prefix}_flow_xy_impact_score"
        high_count_col = f"{prefix}_flow_high_change_cell_count"
        valid_count_col = f"{prefix}_flow_valid_cell_count"
        ratio_col = f"{prefix}_flow_high_change_cell_ratio"
        duration_col = f"{prefix}_flow_event_duration_steps"
        cell_count_col = f"{prefix}_flow_cell_count"
        if not stats or amount_col not in result.columns:
            for col in [amount_score_col, globality_col, transience_col, impact_col, high_count_col, ratio_col, duration_col]:
                result[col] = 0.0
            continue
        amount_score = _score_from_scale(result[amount_col].to_numpy(dtype=float), stats.get("amount_scale", {}))
        result[amount_score_col] = amount_score
        cell_cols = [c for c in stats.get("cell_columns", []) if c in result.columns]
        if cell_cols:
            cell_raw = result[cell_cols].to_numpy(dtype=float)
            cell_scores = _score_from_scale(cell_raw, stats.get("cell_scale", {}))
            valid_matrix = np.isfinite(cell_raw)
            raw_valid_counts = (
                result[valid_count_col].to_numpy(dtype=float)
                if valid_count_col in result.columns
                else valid_matrix.sum(axis=1).astype(float)
            )
            valid_counts = np.minimum(np.nan_to_num(raw_valid_counts, nan=0.0), valid_matrix.sum(axis=1).astype(float))
            high_matrix = (cell_scores >= float(FLOW_EVENT_THRESHOLD)) & valid_matrix
            high_counts = np.minimum(high_matrix.sum(axis=1).astype(float), valid_counts)
            cell_mean_score = np.divide(
                (cell_scores * valid_matrix).sum(axis=1),
                valid_counts,
                out=np.zeros(len(result), dtype=float),
                where=valid_counts > 0,
            )
        else:
            valid_counts = np.zeros(len(result), dtype=float)
            high_counts = np.zeros(len(result), dtype=float)
            cell_mean_score = np.zeros(len(result), dtype=float)
        high_ratio = np.divide(high_counts, valid_counts, out=np.zeros(len(result), dtype=float), where=valid_counts > 0)
        globality = np.clip(
            float(FLOW_GLOBALITY_RATIO_WEIGHT) * high_ratio
            + float(FLOW_GLOBALITY_INTENSITY_WEIGHT) * np.sqrt(np.clip(high_ratio * cell_mean_score, 0.0, 1.0)),
            0.0,
            1.0,
        )
        event_signal = np.maximum(amount_score, globality)
        active = event_signal >= float(FLOW_EVENT_THRESHOLD)
        durations = _duration_by_video(result, active)
        duration_values = durations.to_numpy(dtype=float)
        transience = np.zeros(len(result), dtype=float)
        has_duration = duration_values > 0
        transience[has_duration] = 1.0 / np.power(duration_values[has_duration], float(FLOW_DURATION_ALPHA))
        impact = np.clip(
            (
                float(FLOW_GLOBALITY_WEIGHT) * globality
                + float(FLOW_TRANSIENCE_WEIGHT) * transience
                + float(FLOW_CHANGE_AMOUNT_WEIGHT) * amount_score
            ) / weight_sum,
            0.0,
            1.0,
        )
        result[globality_col] = globality
        result[transience_col] = transience
        result[impact_col] = impact
        result[high_count_col] = high_counts
        result[valid_count_col] = valid_counts
        result[ratio_col] = high_ratio
        result[duration_col] = durations.to_numpy(dtype=float)
        if cell_count_col not in result.columns:
            result[cell_count_col] = valid_counts
    return result


def robust_positive_zscore(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.zeros((0,), dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return np.zeros_like(values, dtype=float)
    median = float(np.median(finite))
    mad = float(np.median(np.abs(finite - median)))
    scale = max(1.4826 * mad, 1e-6)
    return np.maximum((np.nan_to_num(values, nan=median) - median) / scale, 0.0)


def percentile_normalize_0_1(
    values: np.ndarray | pd.Series,
    lower_percentile: float = DIRECTION_JITTER_SCORE_LOWER_PERCENTILE,
    upper_percentile: float = DIRECTION_JITTER_SCORE_UPPER_PERCENTILE,
    positive_only: bool = True,
) -> np.ndarray:
    values_arr = np.asarray(values, dtype=float)
    out = np.zeros_like(values_arr, dtype=float)
    finite_mask = np.isfinite(values_arr)
    if not finite_mask.any():
        return out
    fit_values = values_arr[finite_mask]
    if positive_only:
        fit_values = fit_values[fit_values > FLOW_SCORE_MIN_VISIBLE]
    if fit_values.size == 0:
        return out
    lower_q = float(np.clip(float(lower_percentile), 0.0, 100.0))
    upper_q = float(np.clip(float(upper_percentile), 0.0, 100.0))
    if upper_q < lower_q:
        lower_q, upper_q = upper_q, lower_q
    lower = float(np.percentile(fit_values, lower_q))
    upper = float(np.percentile(fit_values, upper_q))
    if upper <= lower:
        out[finite_mask] = np.where(values_arr[finite_mask] > FLOW_SCORE_MIN_VISIBLE, 1.0, 0.0)
    else:
        out[finite_mask] = np.clip((values_arr[finite_mask] - lower) / max(upper - lower, 1e-6), 0.0, 1.0)
    if positive_only:
        out = np.where(values_arr > FLOW_SCORE_MIN_VISIBLE, out, 0.0)
    return out.astype(float)


def build_flow_window_starts(duration_sec: float, window_sec: float = FLOW_SCORE_WINDOW_SEC, hop_sec: float = FLOW_SCORE_HOP_SEC) -> list[float]:
    max_start = max(float(duration_sec) - float(window_sec), 0.0)
    starts = np.arange(0.0, max_start + 1e-9, float(hop_sec), dtype=np.float32).tolist()
    return [float(value) for value in (starts if starts else [0.0])]


def compute_flow_direction_change_abs(raw_flow_df: pd.DataFrame, x_col: str, y_col: str) -> pd.Series:
    """Compute wrapped flow direction change in radians, ignoring near-zero vectors."""
    if x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    x = ordered[x_col].astype(float).to_numpy()
    y = ordered[y_col].astype(float).to_numpy()
    magnitude = np.hypot(x, y)
    angle = np.arctan2(y, x)
    delta = np.diff(angle, prepend=angle[:1])
    wrapped_delta = np.arctan2(np.sin(delta), np.cos(delta))
    direction_change = np.abs(wrapped_delta)
    valid = (magnitude >= FLOW_DIRECTION_MIN_MAG) & (np.r_[magnitude[:1], magnitude[:-1]] >= FLOW_DIRECTION_MIN_MAG)
    direction_change = np.where(valid, direction_change, 0.0)
    direction_change = np.clip(direction_change, 0.0, np.pi)
    return pd.Series(direction_change, index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


def resolve_direction_jitter_thresholds(
    scores: np.ndarray | pd.Series,
    high_threshold: float = DIRECTION_JITTER_HIGH_THRESHOLD,
    low_threshold: float = DIRECTION_JITTER_LOW_THRESHOLD,
    threshold_mode: str = DIRECTION_JITTER_THRESHOLD_MODE,
    high_percentile: float = DIRECTION_JITTER_HIGH_PERCENTILE,
    low_percentile: float = DIRECTION_JITTER_LOW_PERCENTILE,
) -> tuple[float, float]:
    finite_scores = np.asarray(scores, dtype=float)
    positive_scores = finite_scores[np.isfinite(finite_scores) & (finite_scores > FLOW_SCORE_MIN_VISIBLE)]
    if str(threshold_mode).lower() == "percentile" and positive_scores.size:
        high = float(np.percentile(positive_scores, np.clip(float(high_percentile), 0.0, 100.0)))
        low = float(np.percentile(positive_scores, np.clip(float(low_percentile), 0.0, 100.0)))
    else:
        high = float(high_threshold)
        low = float(low_threshold)
    if low > high:
        low = high
    return high, low


def mark_direction_jitter_hysteresis_windows(
    window_df: pd.DataFrame,
    high_threshold: float = DIRECTION_JITTER_HIGH_THRESHOLD,
    low_threshold: float = DIRECTION_JITTER_LOW_THRESHOLD,
    threshold_mode: str = DIRECTION_JITTER_THRESHOLD_MODE,
    high_percentile: float = DIRECTION_JITTER_HIGH_PERCENTILE,
    low_percentile: float = DIRECTION_JITTER_LOW_PERCENTILE,
) -> pd.DataFrame:
    """Select high-score windows and adjacent low-threshold windows within one grid."""
    window_df = window_df.sort_values("window_start_sec").copy()
    scores = window_df["direction_jitter_score"].to_numpy(dtype=float)
    finite_scores = np.nan_to_num(scores, nan=-np.inf)
    resolved_high_threshold, resolved_low_threshold = resolve_direction_jitter_thresholds(
        scores,
        high_threshold=high_threshold,
        low_threshold=low_threshold,
        threshold_mode=threshold_mode,
        high_percentile=high_percentile,
        low_percentile=low_percentile,
    )
    seed_mask = (finite_scores >= resolved_high_threshold) & (finite_scores > FLOW_SCORE_MIN_VISIBLE)
    low_mask = (finite_scores >= resolved_low_threshold) & (finite_scores > FLOW_SCORE_MIN_VISIBLE)
    selected = np.zeros(len(window_df), dtype=bool)

    for seed_index in np.flatnonzero(seed_mask):
        selected[seed_index] = True
        left = seed_index - 1
        while left >= 0 and low_mask[left]:
            selected[left] = True
            left -= 1
        right = seed_index + 1
        while right < len(window_df) and low_mask[right]:
            selected[right] = True
            right += 1

    window_df["direction_jitter_seed"] = seed_mask.astype(bool)
    window_df["direction_jitter_selected"] = selected.astype(bool)
    window_df["direction_jitter_high_threshold"] = float(resolved_high_threshold)
    window_df["direction_jitter_low_threshold"] = float(resolved_low_threshold)
    return window_df


def add_direction_jitter_score_alpha(window_df: pd.DataFrame) -> pd.DataFrame:
    window_df = window_df.copy()
    if window_df.empty or "direction_jitter_score" not in window_df.columns:
        window_df["direction_jitter_seed"] = False
        window_df["direction_jitter_selected"] = False
        window_df["direction_jitter_score_alpha"] = 0.0
        return window_df

    window_df = mark_direction_jitter_hysteresis_windows(window_df)
    selected_scores = window_df.loc[window_df["direction_jitter_selected"], "direction_jitter_score"].to_numpy(dtype=float)
    finite = selected_scores[np.isfinite(selected_scores)]
    if finite.size == 0 or float(np.nanmax(finite)) <= float(FLOW_SCORE_MIN_VISIBLE):
        window_df["direction_jitter_score_alpha"] = 0.0
        return window_df

    lower = float(np.nanmin(finite))
    upper = float(np.nanmax(finite))
    scores = window_df["direction_jitter_score"].to_numpy(dtype=float)
    if upper <= lower:
        normalized = np.where(window_df["direction_jitter_selected"].to_numpy(dtype=bool), 1.0, 0.0)
    else:
        normalized = np.clip((np.nan_to_num(scores, nan=lower) - lower) / max(upper - lower, 1e-6), 0.0, 1.0)
    alpha = FLOW_SCORE_ALPHA_MIN + normalized * (FLOW_SCORE_ALPHA_MAX - FLOW_SCORE_ALPHA_MIN)
    alpha = np.where(window_df["direction_jitter_selected"].to_numpy(dtype=bool), alpha, 0.0)
    window_df["direction_jitter_score_alpha"] = alpha.astype(float)
    return window_df


def build_flow_direction_jitter_window_table(
    raw_flow_df: pd.DataFrame,
    camera: str,
    gy: int,
    gx: int,
    window_sec: float = FLOW_SCORE_WINDOW_SEC,
    hop_sec: float = FLOW_SCORE_HOP_SEC,
) -> pd.DataFrame:
    x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
    y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
    if raw_flow_df.empty or x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.DataFrame()

    ordered = raw_flow_df.sort_values("time").reset_index(drop=True)
    direction_change = compute_flow_direction_change_abs(ordered, x_col, y_col).to_numpy(dtype=float)
    times = ordered["time"].astype(float).to_numpy()
    if times.size == 0:
        return pd.DataFrame()
    duration_sec = float(np.nanmax(times)) if np.isfinite(times).any() else 0.0
    duration_sec = max(duration_sec, float(window_sec))

    rows = []
    for start_sec in build_flow_window_starts(duration_sec=duration_sec, window_sec=window_sec, hop_sec=hop_sec):
        end_sec = float(min(start_sec + window_sec, duration_sec))
        start_sec = max(end_sec - window_sec, 0.0)
        mask = (times >= start_sec) & (times < end_sec)
        if not np.any(mask):
            nearest_index = int(np.argmin(np.abs(times - start_sec)))
            mask = np.zeros_like(times, dtype=bool)
            mask[nearest_index] = True
        segment = direction_change[mask]
        direction_mean = float(np.mean(segment))
        direction_sum = float(np.sum(segment))
        direction_p95 = float(np.percentile(segment, 95))
        direction_max = float(np.max(segment))
        direction_std = float(np.std(segment))
        direction_variation = float(np.mean(np.abs(np.diff(segment)))) if segment.size >= 2 else 0.0
        if direction_max > FLOW_SCORE_MIN_VISIBLE:
            high_threshold = direction_max * float(FLOW_SCORE_HIGH_RATIO_FRACTION)
            high_ratio = float(np.mean(segment >= high_threshold))
        else:
            high_ratio = 0.0
        rows.append({
            "camera": camera,
            "grid_x": int(gx + 1),
            "grid_y": int(gy + 1),
            "grid_col": int(gx),
            "grid_row": int(gy),
            "window_start_sec": float(start_sec),
            "window_end_sec": float(end_sec),
            "window_center_sec": float(0.5 * (start_sec + end_sec)),
            "direction_mean": direction_mean,
            "direction_sum": direction_sum,
            "direction_p95": direction_p95,
            "direction_max": direction_max,
            "direction_std": direction_std,
            "direction_variation": direction_variation,
            "direction_high_ratio": high_ratio,
        })

    window_df = pd.DataFrame(rows)
    for feature_name in ["direction_sum", "direction_high_ratio", "direction_variation", "direction_p95"]:
        window_df[f"{feature_name}_z"] = robust_positive_zscore(window_df[feature_name].to_numpy(dtype=float))
    window_df["direction_jitter_score_raw"] = (
        0.35 * window_df["direction_sum_z"]
        + 0.25 * window_df["direction_high_ratio_z"]
        + 0.25 * window_df["direction_variation_z"]
        + 0.15 * window_df["direction_p95_z"]
    ).astype(float)
    window_df["direction_jitter_score"] = percentile_normalize_0_1(
        window_df["direction_jitter_score_raw"].to_numpy(dtype=float)
    )
    return add_direction_jitter_score_alpha(window_df)


def build_flow_direction_jitter_score_table(raw_flow_df: pd.DataFrame, camera: str) -> pd.DataFrame:
    rows = []
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            window_df = build_flow_direction_jitter_window_table(raw_flow_df, camera, gy, gx)
            if len(window_df):
                rows.append(window_df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def add_direction_jitter_topk_columns(direction_jitter_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    if direction_jitter_df is None or direction_jitter_df.empty:
        return pd.DataFrame()
    required_cols = {"video_id", "camera", "window_start_sec", "direction_jitter_score"}
    missing_cols = sorted(required_cols - set(direction_jitter_df.columns))
    if missing_cols:
        raise ValueError(f"Missing columns for direction jitter top-k: {missing_cols}")

    work = direction_jitter_df.copy()
    selected = work.get("direction_jitter_selected", pd.Series(False, index=work.index)).astype(bool)
    scores = pd.to_numeric(work["direction_jitter_score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    work["thresholded_direction_jitter_score"] = np.where(selected, scores, 0.0)
    work["direction_jitter_topk_rank"] = pd.Series(pd.NA, index=work.index, dtype="Int64")
    work["direction_jitter_topk_selected"] = False
    work["window_start_bin"] = np.round(work["window_start_sec"].astype(float) / max(float(FLOW_SCORE_HOP_SEC), 1e-6)).astype(int)

    group_cols = ["video_id", "camera", "window_start_bin"]
    for _, group in work.groupby(group_cols, sort=False):
        ranked = group[group["thresholded_direction_jitter_score"].astype(float) > FLOW_SCORE_MIN_VISIBLE]
        ranked = ranked.sort_values("thresholded_direction_jitter_score", ascending=False).head(max(1, int(top_k)))
        if ranked.empty:
            continue
        work.loc[ranked.index, "direction_jitter_topk_rank"] = np.arange(1, len(ranked) + 1, dtype=int)
        work.loc[ranked.index, "direction_jitter_topk_selected"] = True
    return work


def build_broad_vibration_score_table(direction_jitter_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    """Average the top-k thresholded direction-jitter scores per camera for each time window."""
    if direction_jitter_df is None or direction_jitter_df.empty:
        return pd.DataFrame()
    required_cols = {"video_id", "camera", "window_start_sec", "window_end_sec", "window_center_sec", "direction_jitter_score"}
    missing_cols = sorted(required_cols - set(direction_jitter_df.columns))
    if missing_cols:
        raise ValueError(f"Missing columns for broad vibration score: {missing_cols}")

    work = add_direction_jitter_topk_columns(direction_jitter_df, top_k=top_k)
    rows = []
    group_cols = ["video_id", "camera", "window_start_bin"]
    for keys, group in work.groupby(group_cols, sort=True):
        video_id, camera, window_start_bin = keys
        thresholded_scores = group["thresholded_direction_jitter_score"].to_numpy(dtype=float)
        top_values = np.sort(np.nan_to_num(thresholded_scores, nan=0.0, posinf=0.0, neginf=0.0))[::-1]
        top_k_values = np.zeros(max(1, int(top_k)), dtype=float)
        n_values = min(top_k_values.size, top_values.size)
        if n_values:
            top_k_values[:n_values] = top_values[:n_values]
        rows.append({
            "video_id": video_id,
            "camera": camera,
            "window_start_bin": int(window_start_bin),
            "window_start_sec": float(group["window_start_sec"].min()),
            "window_end_sec": float(group["window_end_sec"].max()),
            "window_center_sec": float(group["window_center_sec"].median()),
            "broad_vibration_score": float(top_k_values.mean()),
            "broad_vibration_top_k": int(top_k_values.size),
            "selected_grid_count": int(group.get("direction_jitter_selected", pd.Series(False, index=group.index)).astype(bool).sum()),
            "nonzero_top_k_count": int(np.count_nonzero(top_k_values > FLOW_SCORE_MIN_VISIBLE)),
            "max_thresholded_direction_jitter_score": float(np.max(top_values)) if top_values.size else 0.0,
        })
    return pd.DataFrame(rows).sort_values(["video_id", "camera", "window_start_sec"]).reset_index(drop=True)


def build_broad_vibration_feature_df(raw_flow_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    if raw_flow_df is None or raw_flow_df.empty:
        return pd.DataFrame(columns=["video_id", "time", "time_bin", *BROAD_VIBRATION_COLUMNS])

    raw_flow_df = raw_flow_df.copy()
    if "video_id" not in raw_flow_df.columns:
        raw_flow_df["video_id"] = "unknown"
    direction_tables = []
    for video_id, video_df in raw_flow_df.groupby("video_id", sort=False):
        for camera in ["front", "rear"]:
            direction_df = build_flow_direction_jitter_score_table(video_df, camera)
            if direction_df.empty:
                continue
            direction_df.insert(0, "video_id", video_id)
            direction_tables.append(direction_df)
    if not direction_tables:
        return pd.DataFrame(columns=["video_id", "time", "time_bin", *BROAD_VIBRATION_COLUMNS])

    direction_jitter_df = pd.concat(direction_tables, ignore_index=True)
    broad_df = build_broad_vibration_score_table(direction_jitter_df, top_k=top_k)
    if broad_df.empty:
        return pd.DataFrame(columns=["video_id", "time", "time_bin", *BROAD_VIBRATION_COLUMNS])

    broad_df = broad_df.copy()
    broad_df["time"] = broad_df["window_start_sec"].astype(float)
    broad_df["time_bin"] = np.round(broad_df["time"] / max(float(WINDOW_SEC), 1e-6)).astype(int)
    feature_df = broad_df.pivot_table(
        index=["video_id", "time_bin", "time"],
        columns="camera",
        values="broad_vibration_score",
        aggfunc="mean",
    ).reset_index()
    feature_df = feature_df.rename(columns={
        "front": "front_broad_vibration_score",
        "rear": "rear_broad_vibration_score",
    })
    feature_df.columns.name = None
    for col in BROAD_VIBRATION_COLUMNS:
        if col not in feature_df.columns:
            feature_df[col] = 0.0
    return feature_df[["video_id", "time_bin", "time", *BROAD_VIBRATION_COLUMNS]].copy()


def ensure_broad_vibration_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in BROAD_VIBRATION_COLUMNS:
        if col not in df.columns:
            df[col] = 0.0
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            if "video_id" in df.columns:
                df[col] = df.groupby("video_id")[col].transform(lambda s: s.ffill().bfill()).fillna(0.0)
            else:
                df[col] = df[col].fillna(0.0)
    return df


def extract_sample_features(sample: pd.Series, return_raw_flow: bool = False):
    video_id = str(sample["sample_id"])
    feature_dfs = [extract_audio_features(sample["audio_path"], video_id=video_id)]
    raw_flow_dfs = []
    for prefix, path_col in [("front", "front_path"), ("rear", "rear_path")]:
        frames = extract_video_frames(sample[path_col])
        flow_df = compute_optical_flow_features(frames, prefix, video_id=video_id)
        feature_dfs.append(flow_df)
        raw_flow_cols = [
            "video_id", "time", "time_bin",
            f"{prefix}_flow_x_mean",
            f"{prefix}_flow_y_mean",
            f"{prefix}_flow_x_std",
            f"{prefix}_flow_y_std",
            f"{prefix}_flow_mag_mean",
            f"{prefix}_flow_failed",
        ]
        for gy in range(FLOW_GRID[0]):
            for gx in range(FLOW_GRID[1]):
                raw_flow_cols.extend([
                    f"{prefix}_flow_cell_{gy}_{gx}_x_mean",
                    f"{prefix}_flow_cell_{gy}_{gx}_y_mean",
                    f"{prefix}_flow_cell_{gy}_{gx}_valid_ratio",
                ])
        raw_flow_cols = [c for c in raw_flow_cols if c in flow_df.columns]
        if raw_flow_cols:
            raw_flow_dfs.append(flow_df[raw_flow_cols].copy())
        feature_dfs.append(compute_global_motion_features(frames, prefix, video_id=video_id))
    # 学習時と同じく、センサーCSVなしを示す特徴を全窓へ伝播させる。
    feature_dfs.append(pd.DataFrame({"video_id": [video_id], "time_bin": [0], "time": [0.0], "sensor_missing": [1]}))
    df = align_features_by_time(feature_dfs)
    df = add_flow_change_features(df)
    df = apply_flow_xy_impact_calibration(df, flow_xy_impact_calibration)
    df = add_cross_camera_features(df)

    if raw_flow_dfs:
        raw_flow_df = raw_flow_dfs[0]
        for other in raw_flow_dfs[1:]:
            raw_flow_df = raw_flow_df.merge(other.drop(columns=["time_bin"], errors="ignore"), on=["video_id", "time"], how="outer")
        raw_flow_df = raw_flow_df.sort_values(["video_id", "time"]).reset_index(drop=True)
        raw_flow_df["time_bin"] = np.round(raw_flow_df["time"] / RAW_FLOW_SAMPLE_SEC).astype(int)
        raw_flow_df["target_category"] = sample.get("category", "unknown")
        raw_flow_df["target_environment"] = sample.get("environment", "unknown")
        raw_flow_df["target_label"] = sample.get("plot_label", str(video_id))
    else:
        raw_flow_df = pd.DataFrame(columns=["video_id", "time", "time_bin", "target_category", "target_environment", "target_label"])

    broad_feature_df = build_broad_vibration_feature_df(raw_flow_df)
    if len(broad_feature_df):
        df = df.merge(broad_feature_df.drop(columns=["time"], errors="ignore"), on=["video_id", "time_bin"], how="left")
    df = ensure_broad_vibration_columns(df)
    df["environment"] = sample.get("environment", "unknown")
    df["label"] = sample.get("category", "unknown")

    if not return_raw_flow:
        return df
    return df, raw_flow_df


## 4. 推論と保存

In [109]:
def build_model_input(inference_df: pd.DataFrame, feature_names: np.ndarray) -> pd.DataFrame:
    exclude_cols = {
        "time", "time_bin", "video_id", "sample_id", "label",
        "category", "target_category", "target_environment", "target_label", "plot_label",
        "front_path", "rear_path", "audio_path", "output_dir",
    }
    model_df = inference_df.drop(columns=[c for c in exclude_cols if c in inference_df.columns], errors="ignore")
    model_df = pd.get_dummies(model_df, columns=model_df.select_dtypes(include=["object", "category"]).columns, dummy_na=True)
    return model_df.replace([np.inf, -np.inf], np.nan).reindex(columns=feature_names, fill_value=0.0)


def apply_score_calibration(scores: np.ndarray | pd.Series, calibration: dict[str, float]) -> np.ndarray:
    values = np.asarray(scores, dtype=float)
    lower = float(calibration.get("lower", 0.0))
    upper = float(calibration.get("upper", lower + 1.0))
    denom = max(upper - lower, 1e-6)
    return np.clip((values - lower) / denom, 0.0, 1.0)


def compute_temporal_sync_score(
    scored_df: pd.DataFrame,
    audio_col: str = "audio_anomaly_score",
    motion_col: str = "motion_anomaly_score",
    max_lag_windows: int = 2,
) -> pd.Series:
    if audio_col not in scored_df.columns or motion_col not in scored_df.columns:
        return pd.Series(0.0, index=scored_df.index, name="sync_score")

    sync = pd.Series(0.0, index=scored_df.index, name="sync_score")
    sort_col = "time_bin" if "time_bin" in scored_df.columns else "time"
    window = max(1, int(max_lag_windows) * 2 + 1)

    for _, group in scored_df.sort_values(["video_id", sort_col]).groupby("video_id", sort=False):
        audio = group[audio_col].astype(float).fillna(0.0).clip(0.0, 1.0)
        motion = group[motion_col].astype(float).fillna(0.0).clip(0.0, 1.0)
        nearby_audio = audio.rolling(window=window, center=True, min_periods=1).max()
        nearby_motion = motion.rolling(window=window, center=True, min_periods=1).max()
        aligned = np.maximum(audio * nearby_motion, motion * nearby_audio)
        sync.loc[group.index] = np.sqrt(np.clip(aligned, 0.0, 1.0))
    return sync


def add_composite_scores(
    scored_df: pd.DataFrame,
    sync_config: dict = sync_score_config,
    score_weights: dict[str, float] = final_score_weights,
) -> pd.DataFrame:
    scored_df = scored_df.copy()
    scored_df["sync_score"] = compute_temporal_sync_score(
        scored_df,
        audio_col=sync_config.get("audio_score_column", "audio_anomaly_score"),
        motion_col=sync_config.get("motion_score_column", "motion_anomaly_score"),
        max_lag_windows=int(sync_config.get("max_lag_windows", 2)),
    )

    available_weights = {col: float(weight) for col, weight in score_weights.items() if col in scored_df.columns and float(weight) > 0}
    weight_sum = sum(available_weights.values())
    if weight_sum <= 0:
        scored_df["final_anomaly_score"] = 0.0
    else:
        final_score = np.zeros(len(scored_df), dtype=float)
        for col, weight in available_weights.items():
            final_score += scored_df[col].astype(float).fillna(0.0).to_numpy() * (weight / weight_sum)
        scored_df["final_anomaly_score"] = np.clip(final_score, 0.0, 1.0)

    scored_df["anomaly_score"] = scored_df["final_anomaly_score"]
    scored_df["anomaly_score_smooth"] = scored_df.groupby("video_id")["anomaly_score"].transform(
        lambda s: s.rolling(window=5, center=True, min_periods=1).mean()
    )
    return scored_df


def score_windows(features_df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    scored_df = features_df.copy()
    model_inputs: dict[str, np.ndarray] = {}

    for model_name, artifact in score_model_artifacts.items():
        feature_names = np.asarray(artifact["feature_names"])
        feature_weight_vector = np.asarray(artifact.get("feature_weight_vector", np.ones(len(feature_names))), dtype=float)
        X_scaled = artifact["preprocess_pipeline"].transform(build_model_input(features_df, feature_names))
        X = X_scaled * feature_weight_vector
        raw_scores = -artifact["model"].decision_function(X)
        score_col = artifact.get("score_column", f"{model_name}_anomaly_score")
        raw_score_col = artifact.get("raw_score_column", f"{model_name}_anomaly_score_raw")
        scored_df[raw_score_col] = raw_scores
        scored_df[score_col] = apply_score_calibration(raw_scores, artifact.get("score_calibration", {"lower": 0.0, "upper": 1.0}))
        model_inputs[model_name] = X

    scored_df = add_composite_scores(scored_df)
    return scored_df, model_inputs


target_results: dict[str, dict] = {}
all_feature_dfs = []
all_raw_flow_dfs = []
all_scored_dfs = []

for sample in tqdm(TARGET_SAMPLES, desc="extract targets"):
    sample_id = str(sample["sample_id"])
    output_dir = Path(sample["output_dir"])
    features_df, raw_flow_df = extract_sample_features(sample, return_raw_flow=True)
    features_df = features_df.reset_index(drop=True)
    features_df["target_category"] = sample["category"]
    features_df["target_environment"] = sample["environment"]
    features_df["target_label"] = sample["plot_label"]

    features_path = output_dir / f"{sample_id}_features.csv"
    raw_flow_path = output_dir / f"{sample_id}_raw_flow_0p1s.csv"
    features_df.to_csv(features_path, index=False)
    raw_flow_df.to_csv(raw_flow_path, index=False)

    scored_df = None
    model_inputs: dict[str, np.ndarray] = {}
    if RUN_INFERENCE:
        scored_df, model_inputs = score_windows(features_df)
        scored_df = scored_df.reset_index(drop=True)
        scored_df["target_category"] = sample["category"]
        scored_df["target_environment"] = sample["environment"]
        scored_df["target_label"] = sample["plot_label"]
        scores_path = output_dir / f"{sample_id}_window_scores.csv"
        scored_df.to_csv(scores_path, index=False)
        all_scored_dfs.append(scored_df)
        print(f"{sample_id}: {len(scored_df)} scored windows -> {scores_path}")
    else:
        print(f"{sample_id}: inference skipped, features={len(features_df)} windows, raw_flow={len(raw_flow_df)} rows -> {raw_flow_path}")

    target_results[sample_id] = {
        "sample": sample,
        "features_df": features_df,
        "raw_flow_df": raw_flow_df,
        "scored_df": scored_df,
        "model_inputs": model_inputs,
    }
    all_feature_dfs.append(features_df)
    all_raw_flow_dfs.append(raw_flow_df)

all_target_features_df = pd.concat(all_feature_dfs, ignore_index=True) if all_feature_dfs else pd.DataFrame()
all_target_raw_flow_df = pd.concat(all_raw_flow_dfs, ignore_index=True) if all_raw_flow_dfs else pd.DataFrame()
all_target_features_path = TARGET_COMPARISON_DIR / "all_target_features.csv"
all_target_raw_flow_path = TARGET_COMPARISON_DIR / "all_target_raw_flow_0p1s.csv"
all_target_features_df.to_csv(all_target_features_path, index=False)
all_target_raw_flow_df.to_csv(all_target_raw_flow_path, index=False)
print(f"combined features: {all_target_features_df.shape} -> {all_target_features_path}")
print(f"combined raw flow: {all_target_raw_flow_df.shape} -> {all_target_raw_flow_path}")

default_display_cols = ["video_id", "target_label", "time", "front_flow_x_mean", "front_flow_y_mean", "rear_flow_x_mean", "rear_flow_y_mean"]
display_cols = [c for c in default_display_cols if c in all_target_raw_flow_df.columns]
if display_cols:
    display(all_target_raw_flow_df[display_cols].head(12))

if RUN_INFERENCE and all_scored_dfs:
    all_target_scored_df = pd.concat(all_scored_dfs, ignore_index=True)
    all_target_scores_path = TARGET_COMPARISON_DIR / "all_target_window_scores.csv"
    all_target_scored_df.to_csv(all_target_scores_path, index=False)
    print(f"combined scores: {all_target_scored_df.shape} -> {all_target_scores_path}")
    score_display_cols = ["video_id", "target_label", "time", *[c for c in PLOT_SCORE_COLUMNS if c in all_target_scored_df.columns], "anomaly_score_smooth"]
    display(all_target_scored_df[[c for c in score_display_cols if c in all_target_scored_df.columns]].head(12))
else:
    all_target_scored_df = pd.DataFrame()


001_後進時にトラックに衝突: 73 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/001_後進時にトラックに衝突/001_後進時にトラックに衝突_window_scores.csv
005_後進旋回時トラックに衝突: 75 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/005_後進旋回時トラックに衝突/005_後進旋回時トラックに衝突_window_scores.csv
1001: 72 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/1001/1001_window_scores.csv
1036: 78 scored windows -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/1036/1036_window_scores.csv
combined features: (298, 226) -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/all_target_features.csv
combined raw flow: (589, 72) -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/all_target_raw_flow_0p1s.csv


,video_id,target_label,time,front_flow_x_mean,front_flow_y_mean,rear_flow_x_mean,rear_flow_y_mean
0,001_後進時にトラックに衝突,anomaly/outdoor/001,0.1,-0.101684,0.252642,0.197494,0.126988
1,001_後進時にトラックに衝突,anomaly/outdoor/001,0.2,0.005737,0.151609,0.150024,0.158697
2,001_後進時にトラックに衝突,anomaly/outdoor/001,0.3,0.243214,0.148870,0.078563,0.189648
3,001_後進時にトラックに衝突,anomaly/outdoor/001,0.4,0.486927,0.112444,0.044169,0.220775
4,001_後進時にトラックに衝突,anomaly/outdoor/001,0.5,0.486281,0.017890,0.041505,0.214895
5,001_後進時にトラックに衝突,anomaly/outdoor/001,0.6,0.193496,-0.142631,0.024847,0.188642
6,001_後進時にトラックに衝突,anomaly/outdoor/001,0.7,-0.244384,-0.338273,0.148851,0.162311
7,001_後進時にトラックに衝突,anomaly/outdoor/001,0.8,-0.585418,-0.807035,0.152977,0.121274
8,001_後進時にトラックに衝突,anomaly/outdoor/001,0.9,-0.349856,-0.301836,0.113423,0.198944
9,001_後進時にトラックに衝突,anomaly/outdoor/001,1.0,-0.423777,-0.042812,0.147335,0.142754


combined scores: (298, 234) -> /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/all_target_window_scores.csv


,video_id,target_label,time,anomaly_score,audio_anomaly_score,motion_anomaly_score,sync_score,anomaly_score_smooth
0,001_後進時にトラックに衝突,anomaly/outdoor/001,0.0,0.061851,0.066341,0.000000,0.159989,0.324169
1,001_後進時にトラックに衝突,anomaly/outdoor/001,0.2,0.406057,0.675459,0.000000,0.510503,0.395509
2,001_後進時にトラックに衝突,anomaly/outdoor/001,0.4,0.504598,0.573313,0.385832,0.557829,0.437455
3,001_後進時にトラックに衝突,anomaly/outdoor/001,0.6,0.609531,0.806499,0.385832,0.557829,0.493817
4,001_後進時にトラックに衝突,anomaly/outdoor/001,0.8,0.605239,0.796960,0.385832,0.557829,0.453791
5,001_後進時にトラックに衝突,anomaly/outdoor/001,1.0,0.343659,0.391650,0.231449,0.432046,0.420553
6,001_後進時にトラックに衝突,anomaly/outdoor/001,1.2,0.205929,0.086724,0.231449,0.429483,0.350288
7,001_後進時にトラックに衝突,anomaly/outdoor/001,1.4,0.338406,0.431286,0.231449,0.316600,0.243089
8,001_後進時にトラックに衝突,anomaly/outdoor/001,1.6,0.258206,0.433079,0.000000,0.316600,0.243869
9,001_後進時にトラックに衝突,anomaly/outdoor/001,1.8,0.069243,0.089799,0.000000,0.144166,0.258814


## 5. 比較グラフと上位窓

`PLOT_SCORE_COLUMNS` でスコア系、`PLOT_FEATURE_COLUMNS` で特徴量系の表示列を切り替えます。`anomaly_score` は合成後の最終異常スコアです。

In [110]:
def plot_anomaly_scores(scored_df: pd.DataFrame, save_path: str | Path | None = None):
    label = scored_df["target_label"].iloc[0]
    cols = [c for c in [*PLOT_SCORE_COLUMNS, *PLOT_FEATURE_COLUMNS] if c in scored_df.columns]
    if not cols:
        print("No anomaly plot columns found.")
        return None
    axes = scored_df.plot(
        x="time",
        y=cols,
        subplots=True,
        figsize=(14, 2.2 * len(cols)),
        legend=True,
        title=f"target={label}",
    )
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    return axes


def plot_overlay_metric(
    scored_df: pd.DataFrame,
    metric_col: str,
    ax=None,
):
    if ax is None:
        _, ax = plt.subplots(figsize=(14, 4))
    for _video_id, df in scored_df.groupby("video_id", sort=False):
        if metric_col not in df.columns:
            continue
        df = df.sort_values("time")
        ax.plot(df["time"], df[metric_col], linewidth=1.6, label=df["target_label"].iloc[0])
    ax.set_title(metric_col)
    ax.set_xlabel("time [sec]")
    ax.set_ylabel(metric_col)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    return ax


def plot_overlay_metrics(
    scored_df: pd.DataFrame,
    metric_cols: list[str],
    save_path: str | Path | None = None,
):
    metric_cols = [c for c in metric_cols if c in scored_df.columns]
    if not metric_cols:
        print("No overlay metric columns found.")
        return None
    fig, axes = plt.subplots(len(metric_cols), 1, figsize=(14, 3.0 * len(metric_cols)), sharex=False)
    if len(metric_cols) == 1:
        axes = [axes]
    for ax, metric_col in zip(axes, metric_cols):
        plot_overlay_metric(scored_df, metric_col, ax=ax)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    return axes


def plot_raw_flow_xy(raw_flow_df: pd.DataFrame, camera: str, save_path: str | Path | None = None):
    """Plot per-grid raw flow x/y as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
            y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            if x_col in raw_flow_df.columns:
                ax.plot(raw_flow_df["time"], raw_flow_df[x_col], linewidth=1.0, label="x")
                plotted = True
            if y_col in raw_flow_df.columns:
                ax.plot(raw_flow_df["time"], raw_flow_df[y_col], linewidth=1.0, label="y")
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35)
            title = f"{gx + 1}x{gy + 1}_{camera} mag>={FLOW_DIRECTION_MIN_MAG:g}"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("flow mean")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"raw optical flow {camera} grid x/y 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow x/y columns found.")
    return axes


def robust_positive_zscore(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.zeros((0,), dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return np.zeros_like(values, dtype=float)
    median = float(np.median(finite))
    mad = float(np.median(np.abs(finite - median)))
    scale = max(1.4826 * mad, 1e-6)
    return np.maximum((np.nan_to_num(values, nan=median) - median) / scale, 0.0)


def percentile_normalize_0_1(
    values: np.ndarray | pd.Series,
    lower_percentile: float = DIRECTION_JITTER_SCORE_LOWER_PERCENTILE,
    upper_percentile: float = DIRECTION_JITTER_SCORE_UPPER_PERCENTILE,
    positive_only: bool = True,
) -> np.ndarray:
    values_arr = np.asarray(values, dtype=float)
    out = np.zeros_like(values_arr, dtype=float)
    finite_mask = np.isfinite(values_arr)
    if not finite_mask.any():
        return out
    fit_values = values_arr[finite_mask]
    if positive_only:
        fit_values = fit_values[fit_values > FLOW_SCORE_MIN_VISIBLE]
    if fit_values.size == 0:
        return out
    lower_q = float(np.clip(float(lower_percentile), 0.0, 100.0))
    upper_q = float(np.clip(float(upper_percentile), 0.0, 100.0))
    if upper_q < lower_q:
        lower_q, upper_q = upper_q, lower_q
    lower = float(np.percentile(fit_values, lower_q))
    upper = float(np.percentile(fit_values, upper_q))
    if upper <= lower:
        out[finite_mask] = np.where(values_arr[finite_mask] > FLOW_SCORE_MIN_VISIBLE, 1.0, 0.0)
    else:
        out[finite_mask] = np.clip((values_arr[finite_mask] - lower) / max(upper - lower, 1e-6), 0.0, 1.0)
    if positive_only:
        out = np.where(values_arr > FLOW_SCORE_MIN_VISIBLE, out, 0.0)
    return out.astype(float)


def build_flow_window_starts(duration_sec: float, window_sec: float = FLOW_SCORE_WINDOW_SEC, hop_sec: float = FLOW_SCORE_HOP_SEC) -> list[float]:
    max_start = max(float(duration_sec) - float(window_sec), 0.0)
    starts = np.arange(0.0, max_start + 1e-9, float(hop_sec), dtype=np.float32).tolist()
    return [float(value) for value in (starts if starts else [0.0])]


def add_flow_window_score_alpha(window_df: pd.DataFrame) -> pd.DataFrame:
    window_df = window_df.copy()
    if window_df.empty or "flow_window_score" not in window_df.columns:
        window_df["flow_window_score_alpha"] = 0.0
        return window_df
    scores = window_df["flow_window_score"].to_numpy(dtype=float)
    finite = scores[np.isfinite(scores)]
    if finite.size == 0 or float(np.nanmax(finite)) <= float(FLOW_SCORE_MIN_VISIBLE):
        window_df["flow_window_score_alpha"] = 0.0
        return window_df
    lower = float(np.nanmin(finite))
    upper = float(np.nanmax(finite))
    if upper <= lower:
        normalized = np.where(scores > FLOW_SCORE_MIN_VISIBLE, 1.0, 0.0)
    else:
        normalized = np.clip((np.nan_to_num(scores, nan=lower) - lower) / max(upper - lower, 1e-6), 0.0, 1.0)
    alpha = FLOW_SCORE_ALPHA_MIN + normalized * (FLOW_SCORE_ALPHA_MAX - FLOW_SCORE_ALPHA_MIN)
    alpha = np.where(scores > FLOW_SCORE_MIN_VISIBLE, alpha, 0.0)
    window_df["flow_window_score_alpha"] = alpha.astype(float)
    return window_df


def compute_flow_component_change_abs(raw_flow_df: pd.DataFrame, col: str) -> pd.Series:
    """Compute |d(flow component)/dt| from raw per-grid flow values."""
    if col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    time = ordered["time"].astype(float)
    dt = time.diff().replace([np.inf, -np.inf], np.nan).fillna(RAW_FLOW_SAMPLE_SEC).clip(lower=1e-6)
    change_abs = (ordered[col].astype(float).diff().fillna(0.0) / dt).abs()
    return pd.Series(change_abs.to_numpy(dtype=float), index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


def compute_flow_vector_change_abs(raw_flow_df: pd.DataFrame, x_col: str, y_col: str) -> pd.Series:
    """Compute sqrt((dx/dt)^2 + (dy/dt)^2) from raw per-grid flow vectors."""
    if x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    time = ordered["time"].astype(float)
    dt = time.diff().replace([np.inf, -np.inf], np.nan).fillna(RAW_FLOW_SAMPLE_SEC).clip(lower=1e-6)
    dx_dt = ordered[x_col].astype(float).diff().fillna(0.0) / dt
    dy_dt = ordered[y_col].astype(float).diff().fillna(0.0) / dt
    vector_change_abs = np.hypot(dx_dt.to_numpy(dtype=float), dy_dt.to_numpy(dtype=float))
    vector_change_abs = np.nan_to_num(vector_change_abs, nan=0.0, posinf=0.0, neginf=0.0)
    return pd.Series(vector_change_abs, index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


def compute_flow_magnitude_change_abs(raw_flow_df: pd.DataFrame, x_col: str, y_col: str) -> pd.Series:
    """Compute abs(d(sqrt(x^2 + y^2))/dt) from raw per-grid flow vectors."""
    if x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    time = ordered["time"].astype(float)
    dt = time.diff().replace([np.inf, -np.inf], np.nan).fillna(RAW_FLOW_SAMPLE_SEC).clip(lower=1e-6)
    magnitude = np.hypot(ordered[x_col].astype(float).to_numpy(), ordered[y_col].astype(float).to_numpy())
    magnitude_change_abs = np.abs(pd.Series(magnitude, index=ordered.index).diff().fillna(0.0).to_numpy(dtype=float) / dt.to_numpy(dtype=float))
    magnitude_change_abs = np.nan_to_num(magnitude_change_abs, nan=0.0, posinf=0.0, neginf=0.0)
    return pd.Series(magnitude_change_abs, index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


def lowpass_raw_flow_series(
    values: pd.Series | np.ndarray,
    sample_sec: float = RAW_FLOW_SAMPLE_SEC,
    cutoff_hz: float = RAW_FLOW_MAGNITUDE_LOWPASS_CUTOFF_HZ,
    order: int = RAW_FLOW_MAGNITUDE_LOWPASS_ORDER,
) -> np.ndarray:
    """Low-pass filter a raw flow time series, falling back to a centered mean for short signals."""
    values_arr = np.asarray(values, dtype=float)
    values_arr = np.nan_to_num(values_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if values_arr.size < 3:
        return values_arr
    sample_rate_hz = 1.0 / max(float(sample_sec), 1e-6)
    nyquist_hz = 0.5 * sample_rate_hz
    normalized_cutoff = float(cutoff_hz) / max(nyquist_hz, 1e-6)
    normalized_cutoff = float(np.clip(normalized_cutoff, 1e-4, 0.999))
    filter_order = max(1, int(order))
    b, a = butter(filter_order, normalized_cutoff, btype="low")
    padlen = 3 * max(len(a), len(b))
    if values_arr.size <= padlen:
        window = max(3, min(values_arr.size, int(round(sample_rate_hz / max(float(cutoff_hz), 1e-6)))))
        if window % 2 == 0 and window > 1:
            window -= 1
        return pd.Series(values_arr).rolling(window=window, center=True, min_periods=1).mean().to_numpy(dtype=float)
    return filtfilt(b, a, values_arr)



def build_flow_component_window_table(
    raw_flow_df: pd.DataFrame,
    camera: str,
    component: str,
    gy: int,
    gx: int,
    window_sec: float = FLOW_SCORE_WINDOW_SEC,
    hop_sec: float = FLOW_SCORE_HOP_SEC,
) -> pd.DataFrame:
    value_col = f"{camera}_flow_cell_{gy}_{gx}_{component}_mean"
    if raw_flow_df.empty or value_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.DataFrame()

    ordered = raw_flow_df.sort_values("time").reset_index(drop=True)
    change_abs = compute_flow_component_change_abs(ordered, value_col).to_numpy(dtype=float)
    times = ordered["time"].astype(float).to_numpy()
    if times.size == 0:
        return pd.DataFrame()
    duration_sec = float(np.nanmax(times)) if np.isfinite(times).any() else 0.0
    duration_sec = max(duration_sec, float(window_sec))

    rows = []
    for start_sec in build_flow_window_starts(duration_sec=duration_sec, window_sec=window_sec, hop_sec=hop_sec):
        end_sec = float(min(start_sec + window_sec, duration_sec))
        start_sec = max(end_sec - window_sec, 0.0)
        mask = (times >= start_sec) & (times < end_sec)
        if not np.any(mask):
            nearest_index = int(np.argmin(np.abs(times - start_sec)))
            mask = np.zeros_like(times, dtype=bool)
            mask[nearest_index] = True
        segment = change_abs[mask]
        change_mean = float(np.mean(segment))
        change_p95 = float(np.percentile(segment, 95))
        change_max = float(np.max(segment))
        if change_max > FLOW_SCORE_MIN_VISIBLE:
            high_threshold = change_max * float(FLOW_SCORE_HIGH_RATIO_FRACTION)
            high_ratio = float(np.mean(segment >= high_threshold))
            peakiness = float(np.clip((change_max - change_mean) / max(change_max, 1e-6), 0.0, 1.0))
            distinctiveness = float(np.clip(peakiness * np.sqrt(max(0.0, 1.0 - high_ratio)), 0.0, 1.0))
        else:
            high_ratio = 0.0
            peakiness = 0.0
            distinctiveness = 0.0
        rows.append({
            "camera": camera,
            "component": component,
            "grid_x": int(gx + 1),
            "grid_y": int(gy + 1),
            "grid_col": int(gx),
            "grid_row": int(gy),
            "window_start_sec": float(start_sec),
            "window_end_sec": float(end_sec),
            "window_center_sec": float(0.5 * (start_sec + end_sec)),
            "change_mean": change_mean,
            "change_p95": change_p95,
            "change_max": change_max,
            "change_high_ratio": high_ratio,
            "change_peakiness": peakiness,
            "flow_window_distinctiveness": distinctiveness,
        })

    window_df = pd.DataFrame(rows)
    for feature_name in ["change_mean", "change_p95", "change_max"]:
        window_df[f"{feature_name}_z"] = robust_positive_zscore(window_df[feature_name].to_numpy(dtype=float))
    base_intensity_score = (
        0.20 * window_df["change_mean_z"]
        + 0.50 * window_df["change_p95_z"]
        + 0.30 * window_df["change_max_z"]
    ).astype(float)
    distinctiveness_multiplier = (
        (1.0 - float(FLOW_SCORE_DISTINCTIVENESS_WEIGHT))
        + float(FLOW_SCORE_DISTINCTIVENESS_WEIGHT) * window_df["flow_window_distinctiveness"].astype(float)
    )
    window_df["flow_window_base_intensity_score"] = base_intensity_score
    window_df["flow_window_score"] = (base_intensity_score * distinctiveness_multiplier).astype(float)
    return add_flow_window_score_alpha(window_df)


def build_flow_grid_window_score_table(raw_flow_df: pd.DataFrame, camera: str, component: str) -> pd.DataFrame:
    rows = []
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            window_df = build_flow_component_window_table(raw_flow_df, camera, component, gy, gx)
            if len(window_df):
                rows.append(window_df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def plot_raw_flow_component_change_abs(
    raw_flow_df: pd.DataFrame,
    camera: str,
    component: str,
    save_path: str | Path | None = None,
    window_score_df: pd.DataFrame | None = None,
):
    """Plot per-grid absolute x or y flow change as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None
    if component not in {"x", "y"}:
        raise ValueError("component must be 'x' or 'y'")

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            value_col = f"{camera}_flow_cell_{gy}_{gx}_{component}_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            change_abs = compute_flow_component_change_abs(raw_flow_df, value_col)
            scored_windows = pd.DataFrame()
            if window_score_df is not None and not window_score_df.empty:
                scored_windows = window_score_df[
                    window_score_df["camera"].astype(str).eq(camera)
                    & window_score_df["component"].astype(str).eq(component)
                    & window_score_df["grid_row"].astype(int).eq(gy)
                    & window_score_df["grid_col"].astype(int).eq(gx)
                ]
                for window_row in scored_windows.itertuples(index=False):
                    alpha = float(getattr(window_row, "flow_window_score_alpha", 0.0))
                    if alpha <= 0.0:
                        continue
                    ax.axvspan(
                        float(window_row.window_start_sec),
                        float(window_row.window_end_sec),
                        color="tab:orange",
                        alpha=alpha,
                        zorder=0,
                    )
            if len(change_abs):
                ax.plot(raw_flow_df["time"], change_abs, linewidth=1.0, label=f"|d{component}/dt|", zorder=2)
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera} mag>={FLOW_DIRECTION_MIN_MAG:g}"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            if len(scored_windows):
                title += f" max={scored_windows['flow_window_score'].max():.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel(f"|d{component}/dt|")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"raw optical flow {camera} grid absolute {component}-change 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow {component} columns found for absolute change.")
    return axes


def plot_raw_flow_vector_change_abs(raw_flow_df: pd.DataFrame, camera: str, save_path: str | Path | None = None):
    """Plot per-grid raw flow vector change magnitude as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
            y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            vector_change_abs = compute_flow_vector_change_abs(raw_flow_df, x_col, y_col)
            if len(vector_change_abs):
                ax.plot(raw_flow_df["time"], vector_change_abs, linewidth=1.0, label="|dflow/dt|", zorder=2)
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera}"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("|dflow/dt|")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"raw optical flow {camera} grid vector change 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow x/y columns found for vector change.")
    return axes


def plot_raw_flow_magnitude_change_abs(raw_flow_df: pd.DataFrame, camera: str, save_path: str | Path | None = None):
    """Plot per-grid raw flow magnitude change as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
            y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            magnitude_change_abs = compute_flow_magnitude_change_abs(raw_flow_df, x_col, y_col)
            if len(magnitude_change_abs):
                ax.plot(raw_flow_df["time"], magnitude_change_abs, linewidth=1.0, label="|d|flow|/dt|", zorder=2)
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera}"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("|d|flow|/dt|")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"raw optical flow {camera} grid magnitude change 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow x/y columns found for magnitude change.")
    return axes


def plot_raw_flow_magnitude_change_abs_lowpass(raw_flow_df: pd.DataFrame, camera: str, save_path: str | Path | None = None):
    """Plot low-pass filtered per-grid raw flow magnitude change as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
            y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            magnitude_change_abs = compute_flow_magnitude_change_abs(raw_flow_df, x_col, y_col)
            if len(magnitude_change_abs):
                filtered = lowpass_raw_flow_series(magnitude_change_abs)
                ax.plot(raw_flow_df["time"], filtered, linewidth=1.2, label="lowpass |d|flow|/dt|", zorder=2)
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera} lp<={RAW_FLOW_MAGNITUDE_LOWPASS_CUTOFF_HZ:g}Hz"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            ax.set_title(title, fontsize=10)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("lowpass |d|flow|/dt|")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"low-pass raw optical flow {camera} grid magnitude change 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow x/y columns found for low-pass magnitude change.")
    return axes


def compute_flow_direction_change_abs(raw_flow_df: pd.DataFrame, x_col: str, y_col: str) -> pd.Series:
    """Compute wrapped flow direction change in radians, ignoring near-zero vectors."""
    if x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    x = ordered[x_col].astype(float).to_numpy()
    y = ordered[y_col].astype(float).to_numpy()
    magnitude = np.hypot(x, y)
    angle = np.arctan2(y, x)
    delta = np.diff(angle, prepend=angle[:1])
    wrapped_delta = np.arctan2(np.sin(delta), np.cos(delta))
    direction_change = np.abs(wrapped_delta)
    valid = (magnitude >= FLOW_DIRECTION_MIN_MAG) & (np.r_[magnitude[:1], magnitude[:-1]] >= FLOW_DIRECTION_MIN_MAG)
    direction_change = np.where(valid, direction_change, 0.0)
    direction_change = np.clip(direction_change, 0.0, np.pi)
    return pd.Series(direction_change, index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


def resolve_direction_jitter_thresholds(
    scores: np.ndarray | pd.Series,
    high_threshold: float = DIRECTION_JITTER_HIGH_THRESHOLD,
    low_threshold: float = DIRECTION_JITTER_LOW_THRESHOLD,
    threshold_mode: str = DIRECTION_JITTER_THRESHOLD_MODE,
    high_percentile: float = DIRECTION_JITTER_HIGH_PERCENTILE,
    low_percentile: float = DIRECTION_JITTER_LOW_PERCENTILE,
) -> tuple[float, float]:
    finite_scores = np.asarray(scores, dtype=float)
    positive_scores = finite_scores[np.isfinite(finite_scores) & (finite_scores > FLOW_SCORE_MIN_VISIBLE)]
    if str(threshold_mode).lower() == "percentile" and positive_scores.size:
        high = float(np.percentile(positive_scores, np.clip(float(high_percentile), 0.0, 100.0)))
        low = float(np.percentile(positive_scores, np.clip(float(low_percentile), 0.0, 100.0)))
    else:
        high = float(high_threshold)
        low = float(low_threshold)
    if low > high:
        low = high
    return high, low


def mark_direction_jitter_hysteresis_windows(
    window_df: pd.DataFrame,
    high_threshold: float = DIRECTION_JITTER_HIGH_THRESHOLD,
    low_threshold: float = DIRECTION_JITTER_LOW_THRESHOLD,
    threshold_mode: str = DIRECTION_JITTER_THRESHOLD_MODE,
    high_percentile: float = DIRECTION_JITTER_HIGH_PERCENTILE,
    low_percentile: float = DIRECTION_JITTER_LOW_PERCENTILE,
) -> pd.DataFrame:
    """Select high-score windows and adjacent low-threshold windows within one grid."""
    window_df = window_df.sort_values("window_start_sec").copy()
    scores = window_df["direction_jitter_score"].to_numpy(dtype=float)
    finite_scores = np.nan_to_num(scores, nan=-np.inf)
    resolved_high_threshold, resolved_low_threshold = resolve_direction_jitter_thresholds(
        scores,
        high_threshold=high_threshold,
        low_threshold=low_threshold,
        threshold_mode=threshold_mode,
        high_percentile=high_percentile,
        low_percentile=low_percentile,
    )
    seed_mask = (finite_scores >= resolved_high_threshold) & (finite_scores > FLOW_SCORE_MIN_VISIBLE)
    low_mask = (finite_scores >= resolved_low_threshold) & (finite_scores > FLOW_SCORE_MIN_VISIBLE)
    selected = np.zeros(len(window_df), dtype=bool)

    for seed_index in np.flatnonzero(seed_mask):
        selected[seed_index] = True
        left = seed_index - 1
        while left >= 0 and low_mask[left]:
            selected[left] = True
            left -= 1
        right = seed_index + 1
        while right < len(window_df) and low_mask[right]:
            selected[right] = True
            right += 1

    window_df["direction_jitter_seed"] = seed_mask.astype(bool)
    window_df["direction_jitter_selected"] = selected.astype(bool)
    window_df["direction_jitter_high_threshold"] = float(resolved_high_threshold)
    window_df["direction_jitter_low_threshold"] = float(resolved_low_threshold)
    return window_df


def add_direction_jitter_score_alpha(window_df: pd.DataFrame) -> pd.DataFrame:
    window_df = window_df.copy()
    if window_df.empty or "direction_jitter_score" not in window_df.columns:
        window_df["direction_jitter_seed"] = False
        window_df["direction_jitter_selected"] = False
        window_df["direction_jitter_score_alpha"] = 0.0
        return window_df

    window_df = mark_direction_jitter_hysteresis_windows(window_df)
    selected_scores = window_df.loc[window_df["direction_jitter_selected"], "direction_jitter_score"].to_numpy(dtype=float)
    finite = selected_scores[np.isfinite(selected_scores)]
    if finite.size == 0 or float(np.nanmax(finite)) <= float(FLOW_SCORE_MIN_VISIBLE):
        window_df["direction_jitter_score_alpha"] = 0.0
        return window_df

    lower = float(np.nanmin(finite))
    upper = float(np.nanmax(finite))
    scores = window_df["direction_jitter_score"].to_numpy(dtype=float)
    if upper <= lower:
        normalized = np.where(window_df["direction_jitter_selected"].to_numpy(dtype=bool), 1.0, 0.0)
    else:
        normalized = np.clip((np.nan_to_num(scores, nan=lower) - lower) / max(upper - lower, 1e-6), 0.0, 1.0)
    alpha = FLOW_SCORE_ALPHA_MIN + normalized * (FLOW_SCORE_ALPHA_MAX - FLOW_SCORE_ALPHA_MIN)
    alpha = np.where(window_df["direction_jitter_selected"].to_numpy(dtype=bool), alpha, 0.0)
    window_df["direction_jitter_score_alpha"] = alpha.astype(float)
    return window_df


def build_flow_direction_jitter_window_table(
    raw_flow_df: pd.DataFrame,
    camera: str,
    gy: int,
    gx: int,
    window_sec: float = FLOW_SCORE_WINDOW_SEC,
    hop_sec: float = FLOW_SCORE_HOP_SEC,
) -> pd.DataFrame:
    x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
    y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
    if raw_flow_df.empty or x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.DataFrame()

    ordered = raw_flow_df.sort_values("time").reset_index(drop=True)
    direction_change = compute_flow_direction_change_abs(ordered, x_col, y_col).to_numpy(dtype=float)
    times = ordered["time"].astype(float).to_numpy()
    if times.size == 0:
        return pd.DataFrame()
    duration_sec = float(np.nanmax(times)) if np.isfinite(times).any() else 0.0
    duration_sec = max(duration_sec, float(window_sec))

    rows = []
    for start_sec in build_flow_window_starts(duration_sec=duration_sec, window_sec=window_sec, hop_sec=hop_sec):
        end_sec = float(min(start_sec + window_sec, duration_sec))
        start_sec = max(end_sec - window_sec, 0.0)
        mask = (times >= start_sec) & (times < end_sec)
        if not np.any(mask):
            nearest_index = int(np.argmin(np.abs(times - start_sec)))
            mask = np.zeros_like(times, dtype=bool)
            mask[nearest_index] = True
        segment = direction_change[mask]
        direction_mean = float(np.mean(segment))
        direction_sum = float(np.sum(segment))
        direction_p95 = float(np.percentile(segment, 95))
        direction_max = float(np.max(segment))
        direction_std = float(np.std(segment))
        direction_variation = float(np.mean(np.abs(np.diff(segment)))) if segment.size >= 2 else 0.0
        if direction_max > FLOW_SCORE_MIN_VISIBLE:
            high_threshold = direction_max * float(FLOW_SCORE_HIGH_RATIO_FRACTION)
            high_ratio = float(np.mean(segment >= high_threshold))
        else:
            high_ratio = 0.0
        rows.append({
            "camera": camera,
            "grid_x": int(gx + 1),
            "grid_y": int(gy + 1),
            "grid_col": int(gx),
            "grid_row": int(gy),
            "window_start_sec": float(start_sec),
            "window_end_sec": float(end_sec),
            "window_center_sec": float(0.5 * (start_sec + end_sec)),
            "direction_mean": direction_mean,
            "direction_sum": direction_sum,
            "direction_p95": direction_p95,
            "direction_max": direction_max,
            "direction_std": direction_std,
            "direction_variation": direction_variation,
            "direction_high_ratio": high_ratio,
        })

    window_df = pd.DataFrame(rows)
    for feature_name in ["direction_sum", "direction_high_ratio", "direction_variation", "direction_p95"]:
        window_df[f"{feature_name}_z"] = robust_positive_zscore(window_df[feature_name].to_numpy(dtype=float))
    window_df["direction_jitter_score_raw"] = (
        0.35 * window_df["direction_sum_z"]
        + 0.25 * window_df["direction_high_ratio_z"]
        + 0.25 * window_df["direction_variation_z"]
        + 0.15 * window_df["direction_p95_z"]
    ).astype(float)
    window_df["direction_jitter_score"] = percentile_normalize_0_1(
        window_df["direction_jitter_score_raw"].to_numpy(dtype=float)
    )
    return add_direction_jitter_score_alpha(window_df)


def build_flow_direction_jitter_score_table(raw_flow_df: pd.DataFrame, camera: str) -> pd.DataFrame:
    rows = []
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            window_df = build_flow_direction_jitter_window_table(raw_flow_df, camera, gy, gx)
            if len(window_df):
                rows.append(window_df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def plot_raw_flow_direction_change(
    raw_flow_df: pd.DataFrame,
    camera: str,
    save_path: str | Path | None = None,
    direction_score_df: pd.DataFrame | None = None,
):
    """Plot per-grid wrapped flow direction change as a 3x3 camera map."""
    if raw_flow_df.empty:
        print(f"No raw flow rows found for {camera}.")
        return None

    grid_rows, grid_cols = FLOW_GRID
    label = raw_flow_df["target_label"].iloc[0] if "target_label" in raw_flow_df.columns and len(raw_flow_df) else "raw_flow"
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
            y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{camera}_flow_cell_{gy}_{gx}_valid_ratio"
            scored_windows = pd.DataFrame()
            if direction_score_df is not None and len(direction_score_df):
                scored_windows = direction_score_df[
                    direction_score_df["camera"].astype(str).eq(camera)
                    & direction_score_df["grid_row"].astype(int).eq(gy)
                    & direction_score_df["grid_col"].astype(int).eq(gx)
                ]
                for window_row in scored_windows.itertuples(index=False):
                    alpha = float(getattr(window_row, "direction_jitter_score_alpha", 0.0))
                    if alpha <= 0.0:
                        continue
                    ax.axvspan(
                        float(window_row.window_start_sec),
                        float(window_row.window_end_sec),
                        color="tab:orange",
                        alpha=alpha,
                        zorder=0,
                    )
            direction_change = compute_flow_direction_change_abs(raw_flow_df, x_col, y_col)
            if len(direction_change):
                ax.plot(raw_flow_df["time"], direction_change, linewidth=1.0, label="|dtheta|", zorder=2)
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            ax.axhline(np.pi, color="tab:red", linewidth=0.7, alpha=0.25, linestyle="--", zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera} mag>={FLOW_DIRECTION_MIN_MAG:g}"
            if valid_col in raw_flow_df.columns:
                valid_mean = raw_flow_df[valid_col].astype(float).mean()
                title += f" valid={valid_mean:.2f}"
            if len(scored_windows):
                selected_count = int(scored_windows.get("direction_jitter_selected", pd.Series(dtype=bool)).sum())
                title += f" max={scored_windows['direction_jitter_score'].max():.2f} sel={selected_count}"
            ax.set_title(title, fontsize=10)
            ax.set_ylim(0.0, np.pi)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("|dtheta| [rad]")
            if gy == grid_rows - 1:
                ax.set_xlabel("time [sec]")
            ax.legend(loc="best", fontsize=8)

    fig.suptitle(f"raw optical flow {camera} grid direction change 0.1s: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} per-grid raw flow x/y columns found for direction change.")
    return axes


def topk_mean_zero_padded(values: pd.Series | np.ndarray, k: int) -> float:
    k = max(1, int(k))
    values_arr = np.asarray(values, dtype=float)
    values_arr = np.nan_to_num(values_arr, nan=0.0, posinf=0.0, neginf=0.0)
    values_arr = np.sort(values_arr)[::-1]
    top_values = np.zeros(k, dtype=float)
    n_values = min(k, values_arr.size)
    if n_values:
        top_values[:n_values] = values_arr[:n_values]
    return float(np.mean(top_values))


def add_direction_jitter_topk_columns(direction_jitter_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    if direction_jitter_df is None or direction_jitter_df.empty:
        return pd.DataFrame()

    required_cols = {"video_id", "target_label", "target_category", "target_environment", "camera", "window_start_sec", "direction_jitter_score"}
    missing_cols = sorted(required_cols - set(direction_jitter_df.columns))
    if missing_cols:
        raise ValueError(f"Missing columns for direction jitter top-k: {missing_cols}")

    work = direction_jitter_df.copy()
    selected = work.get("direction_jitter_selected", pd.Series(False, index=work.index)).astype(bool)
    scores = pd.to_numeric(work["direction_jitter_score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    work["thresholded_direction_jitter_score"] = np.where(selected, scores, 0.0)
    work["direction_jitter_topk_rank"] = pd.Series(pd.NA, index=work.index, dtype="Int64")
    work["direction_jitter_topk_selected"] = False
    work["window_start_bin"] = np.round(work["window_start_sec"].astype(float) / max(float(FLOW_SCORE_HOP_SEC), 1e-6)).astype(int)

    group_cols = ["video_id", "target_label", "target_category", "target_environment", "camera", "window_start_bin"]
    for _, group in work.groupby(group_cols, sort=False):
        ranked = group[group["thresholded_direction_jitter_score"].astype(float) > FLOW_SCORE_MIN_VISIBLE]
        ranked = ranked.sort_values("thresholded_direction_jitter_score", ascending=False).head(max(1, int(top_k)))
        if ranked.empty:
            continue
        work.loc[ranked.index, "direction_jitter_topk_rank"] = np.arange(1, len(ranked) + 1, dtype=int)
        work.loc[ranked.index, "direction_jitter_topk_selected"] = True
    return work


def build_broad_vibration_score_table(direction_jitter_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    """Average the top-k thresholded direction-jitter scores per camera for each time window."""
    if direction_jitter_df is None or direction_jitter_df.empty:
        return pd.DataFrame()

    required_cols = {"video_id", "target_label", "target_category", "target_environment", "camera", "window_start_sec", "window_end_sec", "window_center_sec", "direction_jitter_score"}
    missing_cols = sorted(required_cols - set(direction_jitter_df.columns))
    if missing_cols:
        raise ValueError(f"Missing columns for broad vibration score: {missing_cols}")

    work = add_direction_jitter_topk_columns(direction_jitter_df, top_k=top_k)
    selected = work.get("direction_jitter_selected", pd.Series(False, index=work.index)).astype(bool)

    rows = []
    group_cols = ["video_id", "target_label", "target_category", "target_environment", "camera", "window_start_bin"]
    for keys, group in work.groupby(group_cols, sort=True):
        video_id, target_label, target_category, target_environment, camera, window_start_bin = keys
        thresholded_scores = group["thresholded_direction_jitter_score"].to_numpy(dtype=float)
        top_values = np.sort(np.nan_to_num(thresholded_scores, nan=0.0, posinf=0.0, neginf=0.0))[::-1]
        top_k_values = np.zeros(max(1, int(top_k)), dtype=float)
        n_values = min(top_k_values.size, top_values.size)
        if n_values:
            top_k_values[:n_values] = top_values[:n_values]
        row = {
            "video_id": video_id,
            "target_label": target_label,
            "target_category": target_category,
            "target_environment": target_environment,
            "camera": camera,
            "window_start_bin": int(window_start_bin),
            "window_start_sec": float(group["window_start_sec"].min()),
            "window_end_sec": float(group["window_end_sec"].max()),
            "window_center_sec": float(group["window_center_sec"].median()),
            "broad_vibration_score": float(top_k_values.mean()),
            "broad_vibration_top_k": int(top_k_values.size),
            "selected_grid_count": int(selected.loc[group.index].sum()),
            "nonzero_top_k_count": int(np.count_nonzero(top_k_values > FLOW_SCORE_MIN_VISIBLE)),
            "max_thresholded_direction_jitter_score": float(np.max(top_values)) if top_values.size else 0.0,
        }
        for i, value in enumerate(top_k_values, start=1):
            row[f"broad_vibration_top{i}_score"] = float(value)
        rows.append(row)

    return pd.DataFrame(rows).sort_values(["video_id", "camera", "window_start_sec"]).reset_index(drop=True)


def plot_broad_vibration_score(broad_vibration_df: pd.DataFrame, save_path: str | Path | None = None, title: str | None = None):
    if broad_vibration_df is None or broad_vibration_df.empty:
        print("No broad vibration score rows found.")
        return None

    if "camera" in broad_vibration_df.columns:
        camera_values = [c for c in ["front", "rear"] if c in set(broad_vibration_df["camera"].astype(str))]
        camera_values.extend(sorted(set(broad_vibration_df["camera"].astype(str)) - set(camera_values)))
    else:
        camera_values = [None]

    fig, axes = plt.subplots(len(camera_values), 1, figsize=(14, 3.6 * len(camera_values)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, camera in zip(axes, camera_values):
        if camera is None:
            camera_df = broad_vibration_df.copy()
            group_cols = ["video_id"]
            subplot_title = title or f"broad vibration score top{BROAD_VIBRATION_TOP_K}"
        else:
            camera_df = broad_vibration_df[broad_vibration_df["camera"].astype(str).eq(camera)].copy()
            group_cols = ["video_id"]
            subplot_title = f"{camera}"
        for _video_id, df in camera_df.sort_values(["video_id", "window_start_sec"]).groupby(group_cols, sort=False):
            label = df["target_label"].iloc[0] if "target_label" in df.columns else str(_video_id)
            ax.plot(df["window_start_sec"], df["broad_vibration_score"], marker="o", markersize=3, linewidth=1.5, label=label)
        ax.set_title(subplot_title)
        ax.set_ylabel("broad vibration score")
        ax.set_ylim(0.0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best", fontsize=8)
    axes[-1].set_xlabel("window start [sec]")
    if title and len(camera_values) > 1:
        fig.suptitle(title, y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    return ax


def plot_direction_jitter_score_grid(direction_jitter_df: pd.DataFrame, camera: str, save_path: str | Path | None = None):
    """Plot only per-window vibration scores and mark top-k grids for each timestamp."""
    if direction_jitter_df is None or direction_jitter_df.empty:
        print(f"No direction jitter score rows found for {camera}.")
        return None

    camera_df = direction_jitter_df[direction_jitter_df["camera"].astype(str).eq(camera)].copy()
    if camera_df.empty:
        print(f"No direction jitter score rows found for {camera}.")
        return None

    if "thresholded_direction_jitter_score" not in camera_df.columns or "direction_jitter_topk_rank" not in camera_df.columns:
        camera_df = add_direction_jitter_topk_columns(camera_df)

    grid_rows, grid_cols = FLOW_GRID
    label = camera_df["target_label"].iloc[0] if "target_label" in camera_df.columns and len(camera_df) else "direction_jitter"
    rank_colors = {
        1: "tab:red",
        2: "tab:orange",
        3: "gold",
        4: "tab:green",
        5: "tab:purple",
    }
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(4.8 * grid_cols, 3.0 * grid_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(grid_rows, grid_cols)
    plotted = False

    for gy in range(grid_rows):
        for gx in range(grid_cols):
            ax = axes[gy, gx]
            score_df = camera_df[
                camera_df["grid_row"].astype(int).eq(gy)
                & camera_df["grid_col"].astype(int).eq(gx)
            ].sort_values("window_start_sec")
            if len(score_df):
                ax.plot(
                    score_df["window_start_sec"],
                    score_df["thresholded_direction_jitter_score"],
                    linewidth=1.1,
                    marker="o",
                    markersize=2.5,
                    color="tab:blue",
                    label="vibration score",
                    zorder=2,
                )
                topk_df = score_df[score_df.get("direction_jitter_topk_selected", pd.Series(False, index=score_df.index)).astype(bool)]
                topk_ranks = pd.to_numeric(topk_df["direction_jitter_topk_rank"], errors="coerce").fillna(0).astype(int)
                for rank, color in rank_colors.items():
                    rank_df = topk_df[topk_ranks.eq(rank)]
                    if rank_df.empty:
                        continue
                    ax.scatter(
                        rank_df["window_start_sec"],
                        rank_df["thresholded_direction_jitter_score"],
                        s=36,
                        color=color,
                        edgecolors="black",
                        linewidths=0.4,
                        label=f"top {rank}",
                        zorder=4,
                    )
                plotted = True
            ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.35, zorder=1)
            title = f"{gx + 1}x{gy + 1}_{camera}"
            if len(score_df):
                topk_count = int(score_df.get("direction_jitter_topk_selected", pd.Series(False, index=score_df.index)).astype(bool).sum())
                title += f" max={score_df['thresholded_direction_jitter_score'].max():.2f} top={topk_count}"
            ax.set_title(title, fontsize=10)
            ax.set_ylim(0.0, 1.05)
            ax.grid(True, alpha=0.3)
            if gx == 0:
                ax.set_ylabel("vibration score")
            if gy == grid_rows - 1:
                ax.set_xlabel("window start [sec]")
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                by_label = dict(zip(labels, handles))
                ax.legend(by_label.values(), by_label.keys(), loc="best", fontsize=7)

    fig.suptitle(f"thresholded vibration score {camera} grid top{BROAD_VIBRATION_TOP_K}: {label}", y=1.01)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    if not plotted:
        print(f"No {camera} direction jitter score rows found.")
    return axes


def topk_mean(s: pd.Series, k: int = 5) -> float:
    return s.nlargest(min(k, len(s))).mean()


flow_window_score_tables = []
direction_jitter_score_tables = []
broad_vibration_score_tables = []
if PLOT_RAW_FLOW_XY:
    for sample_id, result in target_results.items():
        raw_flow_df = result["raw_flow_df"]
        output_dir = Path(result["sample"]["output_dir"])
        plot_raw_flow_xy(raw_flow_df, "front", output_dir / f"{sample_id}_front_raw_flow_grid_xy_0p1s.png")
        plot_raw_flow_xy(raw_flow_df, "rear", output_dir / f"{sample_id}_rear_raw_flow_grid_xy_0p1s.png")

        sample_direction_jitter_tables = []
        for camera in ["front", "rear"]:
            direction_score_df = build_flow_direction_jitter_score_table(raw_flow_df, camera)
            if len(direction_score_df):
                direction_score_df.insert(0, "video_id", sample_id)
                direction_score_df.insert(1, "target_label", result["sample"].get("plot_label", sample_id))
                direction_score_df.insert(2, "target_category", result["sample"].get("category", "unknown"))
                direction_score_df.insert(3, "target_environment", result["sample"].get("environment", "unknown"))
                sample_direction_jitter_tables.append(direction_score_df)
        sample_direction_jitter_df = pd.concat(sample_direction_jitter_tables, ignore_index=True) if sample_direction_jitter_tables else pd.DataFrame()
        if len(sample_direction_jitter_df):
            sample_direction_jitter_df = add_direction_jitter_topk_columns(sample_direction_jitter_df)
        result["direction_jitter_window_scores_df"] = sample_direction_jitter_df
        if len(sample_direction_jitter_df):
            sample_direction_jitter_path = output_dir / "direction_jitter_window_scores.csv"
            sample_direction_jitter_df.to_csv(sample_direction_jitter_path, index=False)
            direction_jitter_score_tables.append(sample_direction_jitter_df)
            print(f"saved: {sample_direction_jitter_path}")
            sample_broad_vibration_df = build_broad_vibration_score_table(sample_direction_jitter_df)
            result["broad_vibration_score_df"] = sample_broad_vibration_df
            sample_broad_vibration_path = output_dir / f"{sample_id}_broad_vibration_score_top{BROAD_VIBRATION_TOP_K}.csv"
            sample_broad_vibration_plot_path = output_dir / f"{sample_id}_broad_vibration_score_top{BROAD_VIBRATION_TOP_K}.png"
            sample_broad_vibration_df.to_csv(sample_broad_vibration_path, index=False)
            broad_vibration_score_tables.append(sample_broad_vibration_df)
            plot_broad_vibration_score(
                sample_broad_vibration_df,
                sample_broad_vibration_plot_path,
                title=f"broad vibration score by camera top{BROAD_VIBRATION_TOP_K}: {result['sample'].get('plot_label', sample_id)}",
            )
            print(f"saved: {sample_broad_vibration_path}")
            plot_direction_jitter_score_grid(
                sample_direction_jitter_df,
                "front",
                output_dir / f"{sample_id}_front_vibration_score_grid_top{BROAD_VIBRATION_TOP_K}.png",
            )
            plot_direction_jitter_score_grid(
                sample_direction_jitter_df,
                "rear",
                output_dir / f"{sample_id}_rear_vibration_score_grid_top{BROAD_VIBRATION_TOP_K}.png",
            )
        else:
            result["broad_vibration_score_df"] = pd.DataFrame()

        plot_raw_flow_direction_change(raw_flow_df, "front", output_dir / f"{sample_id}_front_raw_flow_grid_direction_change_0p1s.png", sample_direction_jitter_df)
        plot_raw_flow_direction_change(raw_flow_df, "rear", output_dir / f"{sample_id}_rear_raw_flow_grid_direction_change_0p1s.png", sample_direction_jitter_df)

        sample_window_score_tables = []
        for camera in ["front", "rear"]:
            for component in ["x", "y"]:
                window_score_df = build_flow_grid_window_score_table(raw_flow_df, camera, component)
                if len(window_score_df):
                    window_score_df.insert(0, "video_id", sample_id)
                    window_score_df.insert(1, "target_label", result["sample"].get("plot_label", sample_id))
                    window_score_df.insert(2, "target_category", result["sample"].get("category", "unknown"))
                    window_score_df.insert(3, "target_environment", result["sample"].get("environment", "unknown"))
                    sample_window_score_tables.append(window_score_df)
        sample_window_score_df = pd.concat(sample_window_score_tables, ignore_index=True) if sample_window_score_tables else pd.DataFrame()
        result["flow_window_scores_df"] = sample_window_score_df
        if len(sample_window_score_df):
            sample_window_score_path = output_dir / f"{sample_id}_flow_grid_window_scores.csv"
            sample_window_score_df.to_csv(sample_window_score_path, index=False)
            flow_window_score_tables.append(sample_window_score_df)
            print(f"saved: {sample_window_score_path}")

        plot_raw_flow_component_change_abs(raw_flow_df, "front", "x", output_dir / f"{sample_id}_front_raw_flow_grid_x_change_abs_0p1s.png", sample_window_score_df)
        plot_raw_flow_component_change_abs(raw_flow_df, "front", "y", output_dir / f"{sample_id}_front_raw_flow_grid_y_change_abs_0p1s.png", sample_window_score_df)
        plot_raw_flow_component_change_abs(raw_flow_df, "rear", "x", output_dir / f"{sample_id}_rear_raw_flow_grid_x_change_abs_0p1s.png", sample_window_score_df)
        plot_raw_flow_component_change_abs(raw_flow_df, "rear", "y", output_dir / f"{sample_id}_rear_raw_flow_grid_y_change_abs_0p1s.png", sample_window_score_df)
        plot_raw_flow_vector_change_abs(raw_flow_df, "front", output_dir / f"{sample_id}_front_raw_flow_grid_vector_change_abs_0p1s.png")
        plot_raw_flow_vector_change_abs(raw_flow_df, "rear", output_dir / f"{sample_id}_rear_raw_flow_grid_vector_change_abs_0p1s.png")
        plot_raw_flow_magnitude_change_abs(raw_flow_df, "front", output_dir / f"{sample_id}_front_raw_flow_grid_magnitude_change_abs_0p1s.png")
        plot_raw_flow_magnitude_change_abs(raw_flow_df, "rear", output_dir / f"{sample_id}_rear_raw_flow_grid_magnitude_change_abs_0p1s.png")
        plot_raw_flow_magnitude_change_abs_lowpass(raw_flow_df, "front", output_dir / f"{sample_id}_front_raw_flow_grid_magnitude_change_abs_lowpass_0p1s.png")
        plot_raw_flow_magnitude_change_abs_lowpass(raw_flow_df, "rear", output_dir / f"{sample_id}_rear_raw_flow_grid_magnitude_change_abs_lowpass_0p1s.png")

flow_grid_window_scores_df = pd.concat(flow_window_score_tables, ignore_index=True) if flow_window_score_tables else pd.DataFrame()
if len(flow_grid_window_scores_df):
    flow_window_score_path = TARGET_COMPARISON_DIR / "all_target_flow_grid_window_scores.csv"
    flow_grid_window_scores_df.to_csv(flow_window_score_path, index=False)
    display(flow_grid_window_scores_df.sort_values("flow_window_score", ascending=False).head(30))
    print(f"saved flow window scores: {flow_window_score_path}")
else:
    flow_grid_window_scores_df = pd.DataFrame()

direction_jitter_window_scores_df = pd.concat(direction_jitter_score_tables, ignore_index=True) if direction_jitter_score_tables else pd.DataFrame()
if len(direction_jitter_window_scores_df):
    direction_jitter_path = TARGET_COMPARISON_DIR / "all_target_direction_jitter_window_scores.csv"
    direction_jitter_window_scores_df.to_csv(direction_jitter_path, index=False)
    display(direction_jitter_window_scores_df.sort_values("direction_jitter_score", ascending=False).head(30))
    print(f"saved direction jitter window scores: {direction_jitter_path}")
else:
    direction_jitter_window_scores_df = pd.DataFrame()

broad_vibration_scores_df = pd.concat(broad_vibration_score_tables, ignore_index=True) if broad_vibration_score_tables else pd.DataFrame()
if len(broad_vibration_scores_df):
    broad_vibration_path = TARGET_COMPARISON_DIR / f"all_target_broad_vibration_score_top{BROAD_VIBRATION_TOP_K}.csv"
    broad_vibration_plot_path = TARGET_COMPARISON_DIR / f"overlay_broad_vibration_score_top{BROAD_VIBRATION_TOP_K}.png"
    broad_vibration_scores_df.to_csv(broad_vibration_path, index=False)
    plot_broad_vibration_score(
        broad_vibration_scores_df,
        broad_vibration_plot_path,
        title=f"broad vibration score by camera top{BROAD_VIBRATION_TOP_K} overlay",
    )
    display(broad_vibration_scores_df.sort_values("broad_vibration_score", ascending=False).head(30))
    print(f"saved broad vibration scores: {broad_vibration_path}")
    print(f"saved broad vibration plot  : {broad_vibration_plot_path}")
else:
    broad_vibration_scores_df = pd.DataFrame()

if RUN_INFERENCE and PLOT_EXISTING_GRAPHS and len(all_target_scored_df):
    summary_agg = {
        "max_anomaly_score": ("anomaly_score", "max"),
        "top5_mean_anomaly_score": ("anomaly_score", topk_mean),
        "p95_anomaly_score": ("anomaly_score", lambda s: s.quantile(0.95)),
        "n_windows": ("anomaly_score", "size"),
    }
    for col in ["audio_anomaly_score", "motion_anomaly_score", "sync_score"]:
        if col in all_target_scored_df.columns:
            summary_agg[f"max_{col}"] = (col, "max")
            summary_agg[f"top5_mean_{col}"] = (col, topk_mean)

    target_video_summary = (
        all_target_scored_df
        .groupby(["video_id", "target_category", "target_environment", "target_label"], as_index=False)
        .agg(**summary_agg)
        .sort_values("max_anomaly_score", ascending=False)
    )

    score_cols = [c for c in PLOT_SCORE_COLUMNS if c in all_target_scored_df.columns]
    feature_cols = [c for c in PLOT_FEATURE_COLUMNS if c in all_target_scored_df.columns]
    top_cols = [
        "video_id", "target_category", "target_environment", "target_label", "time",
        *score_cols, "final_anomaly_score", "anomaly_score_smooth", *feature_cols,
    ]
    top_cols = list(dict.fromkeys([c for c in top_cols if c in all_target_scored_df.columns]))
    target_top_windows = pd.concat(
        [g.nlargest(TOP_N_ANOMALIES, "anomaly_score") for _, g in all_target_scored_df.groupby("video_id", sort=False)],
        ignore_index=True,
    )[top_cols]

    key_metric_cols = [
        "video_id", "target_category", "target_environment", "target_label", "time",
        *score_cols, "final_anomaly_score", "anomaly_score_smooth", *feature_cols,
    ]
    key_metric_cols = list(dict.fromkeys([c for c in key_metric_cols if c in all_target_scored_df.columns]))
    all_target_key_metrics_df = all_target_scored_df[key_metric_cols].copy()

    summary_path = TARGET_COMPARISON_DIR / "target_video_scores.csv"
    top_windows_path = TARGET_COMPARISON_DIR / "target_top_windows.csv"
    key_metrics_path = TARGET_COMPARISON_DIR / "all_target_key_metrics.csv"
    overlay_plot_path = TARGET_COMPARISON_DIR / "overlay_anomaly_scores.png"
    overlay_key_metrics_plot_path = TARGET_COMPARISON_DIR / "overlay_key_metrics.png"

    target_video_summary.to_csv(summary_path, index=False)
    target_top_windows.to_csv(top_windows_path, index=False)
    all_target_key_metrics_df.to_csv(key_metrics_path, index=False)
    plot_overlay_metrics(all_target_scored_df, score_cols, overlay_plot_path)
    plot_overlay_metrics(all_target_scored_df, feature_cols, overlay_key_metrics_plot_path)

    for sample_id, result in target_results.items():
        output_dir = Path(result["sample"]["output_dir"])
        scored_df = result["scored_df"]
        target_video_summary[target_video_summary["video_id"].astype(str).eq(str(sample_id))].to_csv(output_dir / f"{sample_id}_video_score.csv", index=False)
        scored_df.nlargest(TOP_N_ANOMALIES, "anomaly_score").to_csv(output_dir / f"{sample_id}_top_windows.csv", index=False)
        plot_anomaly_scores(scored_df, output_dir / f"{sample_id}_anomaly_plot.png")

    display(target_video_summary)
    display(target_top_windows)
    display(all_target_key_metrics_df.head(12))
    print(f"saved summary            : {summary_path}")
    print(f"saved top windows        : {top_windows_path}")
    print(f"saved key metrics        : {key_metrics_path}")
    print(f"saved score overlay      : {overlay_plot_path}")
    print(f"saved feature overlay    : {overlay_key_metrics_plot_path}")
else:
    target_video_summary = pd.DataFrame()
    target_top_windows = pd.DataFrame()
    all_target_key_metrics_df = pd.DataFrame()
    if not RUN_INFERENCE:
        print("Existing anomaly plots skipped because RUN_INFERENCE=False.")
    elif not PLOT_EXISTING_GRAPHS:
        print("Existing anomaly plots skipped because PLOT_EXISTING_GRAPHS=False.")


saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/001_後進時にトラックに衝突/001_後進時にトラックに衝突_front_raw_flow_grid_xy_0p1s.png
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/001_後進時にトラックに衝突/001_後進時にトラックに衝突_rear_raw_flow_grid_xy_0p1s.png
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/001_後進時にトラックに衝突/direction_jitter_window_scores.csv
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/001_後進時にトラックに衝突/001_後進時にトラックに衝突_broad_vibration_score_top5.png
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/001_後進時にトラックに衝突/001_後進時にトラックに衝突_broad_vibration_score_top5.csv
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/001_後進時にトラックに衝突/001_後進時にトラックに衝突_front_vibration_score_grid_top5.png
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/001_後進時にトラックに衝突/001_後進時にトラックに衝突_rear_vibration_score_grid_top5.png
saved: /workspaces/forklift/outputs/anomaly_featur

,video_id,target_label,target_category,target_environment,camera,component,grid_x,grid_y,grid_col,grid_row,window_start_sec,window_end_sec,window_center_sec,change_mean,change_p95,change_max,change_high_ratio,change_peakiness,flow_window_distinctiveness,change_mean_z,change_p95_z,change_max_z,flow_window_base_intensity_score,flow_window_score,flow_window_score_alpha
938,001_後進時にトラックに衝突,anomaly/outdoor/001,anomaly,outdoor,rear,y,1,3,0,2,7.0,8.0,7.5,6.614359,31.830470,32.270977,0.2,0.795037,0.711103,25.663025,47.885114,41.864062,41.634380,31.410527,0.420000
939,001_後進時にトラックに衝突,anomaly/outdoor/001,anomaly,outdoor,rear,y,1,3,0,2,7.5,8.5,8.0,6.727966,31.830470,32.270977,0.2,0.791517,0.707954,26.123159,47.885114,41.864062,41.726407,31.368277,0.419489
482,001_後進時にトラックに衝突,anomaly/outdoor/001,anomaly,outdoor,front,y,3,3,2,2,3.0,4.0,3.5,4.002825,17.570929,18.755236,0.2,0.786576,0.703535,9.769259,19.722952,18.813297,17.459317,13.059646,0.420000
483,001_後進時にトラックに衝突,anomaly/outdoor/001,anomaly,outdoor,front,y,3,3,2,2,3.5,4.5,4.0,4.093258,17.570929,18.755236,0.2,0.781754,0.699222,10.016159,19.722952,18.813297,17.508697,13.032399,0.419207
912,001_後進時にトラックに衝突,anomaly/outdoor/001,anomaly,outdoor,rear,y,3,2,2,1,8.0,9.0,8.5,7.633564,21.348565,26.176645,0.2,0.708383,0.633597,13.339829,18.443061,20.674785,18.091932,12.457330,0.420000
913,001_後進時にトラックに衝突,anomaly/outdoor/001,anomaly,outdoor,rear,y,3,2,2,1,8.5,9.5,9.0,9.652399,21.348565,26.176645,0.2,0.631259,0.564615,17.164549,18.443061,20.674785,18.856875,11.878380,0.402340
2307,1001,normal/outdoor/1001,normal,outdoor,front,y,2,1,1,0,10.5,11.5,11.0,10.397854,31.447877,31.461711,0.2,0.669508,0.598826,9.922815,16.646297,18.991013,16.005016,10.547337,0.420000
577,001_後進時にトラックに衝突,anomaly/outdoor/001,anomaly,outdoor,rear,x,3,1,2,0,8.5,9.5,9.0,6.649788,19.946667,26.350727,0.1,0.747643,0.709277,7.800043,13.465583,18.853115,13.948734,10.501793,0.420000
576,001_後進時にトラックに衝突,anomaly/outdoor/001,anomaly,outdoor,rear,x,3,1,2,0,8.0,9.0,8.5,7.001493,18.935482,26.350727,0.1,0.734296,0.696614,8.275199,12.724155,18.853115,13.673052,10.147075,0.407165
2306,1001,normal/outdoor/1001,normal,outdoor,front,y,2,1,1,0,10.0,11.0,10.5,12.727423,31.447877,31.461711,0.2,0.595463,0.532598,12.590648,16.646297,18.991013,16.538582,9.967946,0.399126


saved flow window scores: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/all_target_flow_grid_window_scores.csv


,video_id,target_label,target_category,target_environment,camera,grid_x,grid_y,grid_col,grid_row,window_start_sec,window_end_sec,window_center_sec,direction_mean,direction_sum,direction_p95,direction_max,direction_std,direction_variation,direction_high_ratio,direction_sum_z,direction_high_ratio_z,direction_variation_z,direction_p95_z,direction_jitter_score_raw,direction_jitter_score,direction_jitter_seed,direction_jitter_selected,direction_jitter_high_threshold,direction_jitter_low_threshold,direction_jitter_score_alpha,thresholded_direction_jitter_score,direction_jitter_topk_rank,direction_jitter_topk_selected,window_start_bin
26,001_後進時にトラックに衝突,anomaly/outdoor/001,anomaly,outdoor,front,1,1,0,0,13.0,14.0,13.5,0.634534,6.345335,2.488368,2.822093,0.932093,1.053592,0.200000,1.525664e+01,0.000000,54.799017,2.412942e+01,2.265899e+01,1.0,True,True,0.724866,0.168183,0.42,1.0,1,True,26
929,005_後進旋回時トラックに衝突,anomaly/outdoor/005,anomaly,outdoor,rear,1,3,0,2,2.5,3.5,3.0,0.429400,4.294004,2.020337,2.780177,0.843727,0.428987,0.100000,4.294004e+06,100000.000000,428987.136869,2.020337e+06,1.938199e+06,1.0,True,True,0.929296,0.796551,0.42,1.0,3,True,5
1171,1001,normal/outdoor/1001,normal,outdoor,front,1,3,0,2,0.5,1.5,1.0,1.344084,13.440844,2.459187,2.481443,0.839426,1.011505,0.600000,2.400794e+00,2.023472,0.844473,8.222595e-01,1.680603e+00,1.0,True,True,0.874909,0.667178,0.42,1.0,1,True,1
1947,1036,normal/indoor/1036,normal,indoor,rear,1,3,0,2,9.0,10.0,9.5,0.528624,5.286240,2.032165,2.626445,0.831102,0.790381,0.100000,5.286240e+06,100000.000000,790381.332554,2.032165e+06,2.377604e+06,1.0,True,True,0.984129,0.623164,0.42,1.0,1,True,18
1977,1036,normal/indoor/1036,normal,indoor,rear,2,3,1,2,9.5,10.5,10.0,0.612275,6.122754,2.096177,2.178978,0.871174,0.591183,0.300000,6.122754e+06,300000.000000,591182.588148,2.096177e+06,2.680186e+06,1.0,True,True,0.892171,0.310993,0.42,1.0,1,True,19
45,001_後進時にトラックに衝突,anomaly/outdoor/001,anomaly,outdoor,front,2,1,1,0,8.5,9.5,9.0,1.342914,13.429138,2.705717,3.077129,1.017697,1.498764,0.500000,3.067208e+00,2.636646,2.225425,1.263106e+00,2.478506e+00,1.0,True,True,0.804762,0.447954,0.42,1.0,1,True,17
1132,1001,normal/outdoor/1001,normal,outdoor,front,2,2,1,1,8.0,9.0,8.5,1.352353,13.523530,2.741252,3.011345,0.917149,0.841335,0.400000,1.924863e+00,1.348982,0.926983,1.438698e+00,1.458498e+00,1.0,True,True,0.810180,0.484644,0.42,1.0,3,True,16
663,005_後進旋回時トラックに衝突,anomaly/outdoor/005,anomaly,outdoor,front,3,2,2,1,9.5,10.5,10.0,1.164638,11.646377,2.270883,2.326857,0.792947,1.082459,0.500000,1.301386e+01,2.023472,14.256117,9.009077e+00,9.976108e+00,1.0,True,True,0.475754,0.277932,0.42,1.0,4,True,19
633,005_後進旋回時トラックに衝突,anomaly/outdoor/005,anomaly,outdoor,front,2,2,1,1,8.5,9.5,9.0,1.595733,15.957333,2.809002,2.862383,1.118615,1.000631,0.600000,3.030797e+00,2.023472,1.133833,1.221967e+00,2.033400e+00,1.0,True,True,0.756546,0.216107,0.42,1.0,1,True,17
662,005_後進旋回時トラックに衝突,anomaly/outdoor/005,anomaly,outdoor,front,3,2,2,1,9.0,10.0,9.5,1.024239,10.242388,2.422551,2.602615,0.922036,1.139085,0.400000,1.132050e+01,1.348982,15.049092,9.672777e+00,9.512610e+00,1.0,True,True,0.475754,0.277932,0.42,1.0,4,True,18


saved direction jitter window scores: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/all_target_direction_jitter_window_scores.csv
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/overlay_broad_vibration_score_top5.png


...

saved broad vibration scores: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/all_target_broad_vibration_score_top5.csv
saved broad vibration plot  : /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/overlay_broad_vibration_score_top5.png
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/overlay_anomaly_scores.png
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/overlay_key_metrics.png
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/001_後進時にトラックに衝突/001_後進時にトラックに衝突_anomaly_plot.png
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/005_後進旋回時トラックに衝突/005_後進旋回時トラックに衝突_anomaly_plot.png
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/1001/1001_anomaly_plot.png
saved: /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/1036/1036_anomaly_plot.png


,video_id,target_category,target_environment,target_label,max_anomaly_score,top5_mean_anomaly_score,p95_anomaly_score,n_windows,max_audio_anomaly_score,top5_mean_audio_anomaly_score,max_motion_anomaly_score,top5_mean_motion_anomaly_score,max_sync_score,top5_mean_sync_score
0,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,0.965875,0.883560,0.823897,73,1.000000,0.880111,1.000000,0.961000,1.000000,0.980000
1,005_後進旋回時トラックに衝突,anomaly,outdoor,anomaly/outdoor/005,0.838915,0.716273,0.560687,75,0.794048,0.719434,0.937895,0.920740,0.844074,0.826111
2,1001,normal,outdoor,normal/outdoor/1001,0.833873,0.770603,0.707183,72,1.000000,1.000000,0.830145,0.784245,0.833213,0.826935
3,1036,normal,indoor,normal/indoor/1036,0.534839,0.455487,0.398300,78,1.000000,0.844477,0.495028,0.428970,0.573722,0.513069


,video_id,target_category,target_environment,target_label,time,anomaly_score,audio_anomaly_score,motion_anomaly_score,sync_score,final_anomaly_score,anomaly_score_smooth,audio_rms,audio_high_freq_energy,front_broad_vibration_score,rear_broad_vibration_score,front_global_translation_norm,rear_global_translation_norm,front_flow_mag_mean,rear_flow_mag_mean,front_flow_x_mean,rear_flow_x_mean,front_flow_y_mean,rear_flow_y_mean,front_flow_change_amount_score,rear_flow_change_amount_score,front_flow_globality_score,rear_flow_globality_score,front_flow_transience_score,rear_flow_transience_score,front_flow_xy_impact_score,rear_flow_xy_impact_score,front_flow_high_change_cell_ratio,rear_flow_high_change_cell_ratio,front_flow_event_duration_steps,rear_flow_event_duration_steps
0,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,8.8,0.965875,1.000000,0.902501,1.000000,0.965875,0.820947,0.348946,0.230505,0.973919,0.757344,1.203057,0.597353,2.033136,2.176952,-0.378871,1.409248,-0.533636,0.427833,0.229496,0.000000,0.226848,0.000000,0.0,0.000000,0.159323,0.000000,0.222222,0.000000,0.0,0.0
1,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,9.0,0.896600,0.770222,1.000000,1.000000,0.896600,0.883560,0.195061,0.058878,0.829999,0.971412,0.679007,0.476739,1.844255,2.060742,-0.820709,1.582371,-1.097762,0.379146,0.255129,0.027431,0.117072,0.000000,0.0,0.000000,0.109562,0.005486,0.111111,0.000000,0.0,0.0
2,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,8.6,0.888470,0.850210,0.902501,0.950001,0.888470,0.752984,0.360856,0.125223,0.973919,0.757344,1.229482,1.815383,1.562741,2.506510,-0.213606,1.467203,0.000395,0.383930,0.297501,0.053895,0.227146,0.000000,0.0,0.000000,0.173073,0.010779,0.222222,0.000000,0.0,0.0
3,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,9.4,0.881077,0.786114,1.000000,0.886631,0.881077,0.772174,0.212888,0.007951,0.829999,0.971412,0.239981,0.394749,1.024319,1.207802,-0.730645,0.756039,-0.308191,-0.006228,0.156584,0.000000,0.115058,0.000000,0.0,0.000000,0.088846,0.000000,0.111111,0.000000,0.0,0.0
4,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,9.2,0.785777,0.523950,1.000000,1.000000,0.785777,0.831579,0.041446,0.615211,0.829999,0.971412,0.482871,0.508918,1.709331,1.598668,-1.164219,1.014990,0.528993,0.229976,0.288977,0.000000,0.115955,0.000000,0.0,0.000000,0.115773,0.000000,0.111111,0.000000,0.0,0.0
5,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,10.2,0.727199,0.818755,0.552387,0.827118,0.727199,0.545408,0.279035,0.018602,0.658285,0.325564,0.024008,0.205568,1.605921,0.362446,0.780009,-0.117765,0.860077,0.082015,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
6,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,9.8,0.668849,0.468837,0.835566,0.827118,0.668849,0.682626,0.153274,0.000066,0.704566,0.835057,0.116175,0.093331,1.023632,0.482828,0.083402,0.099994,-0.752371,0.308810,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
7,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,12.6,0.662974,0.627333,0.707066,0.666007,0.662974,0.494469,0.230014,0.002907,0.439536,0.788328,2.281614,5.509065,1.679790,1.208352,0.822590,1.030108,0.606301,-0.159981,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
8,005_後進旋回時トラックに衝突,anomaly,outdoor,anomaly/outdoor/005,9.2,0.838915,0.759639,0.937895,0.844074,0.838915,0.697801,0.153934,0.130953,0.998277,0.489725,1.374886,1.084230,2.405176,3.572038,-1.433066,-1.265303,0.241593,-0.574355,0.527179,0.362950,0.342617,0.454987,0.0,0.000000,0.276744,0.300084,0.333333,0.444444,0.0,0.0
9,005_後進旋回時トラックに衝突,anomaly,outdoor,anomaly/outdoor/005,9.0,0.805881,0.686229,0.937895,0.844074,0.805881,0.653340,0.163046,0.120191,0.998277,0.489725,1.159928,0.425774,5.558457,3.523336,-3.277656,-1.186216,-0.408742,-0.441052,0.628109,0.555406,0.227766,0.343745,0.0,0.000000,0.239505,0.282954,0.222222,0.333333,0.0,0.0


,video_id,target_category,target_environment,target_label,time,anomaly_score,audio_anomaly_score,motion_anomaly_score,sync_score,final_anomaly_score,anomaly_score_smooth,audio_rms,audio_high_freq_energy,front_broad_vibration_score,rear_broad_vibration_score,front_global_translation_norm,rear_global_translation_norm,front_flow_mag_mean,rear_flow_mag_mean,front_flow_x_mean,rear_flow_x_mean,front_flow_y_mean,rear_flow_y_mean,front_flow_change_amount_score,rear_flow_change_amount_score,front_flow_globality_score,rear_flow_globality_score,front_flow_transience_score,rear_flow_transience_score,front_flow_xy_impact_score,rear_flow_xy_impact_score,front_flow_high_change_cell_ratio,rear_flow_high_change_cell_ratio,front_flow_event_duration_steps,rear_flow_event_duration_steps
0,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,0.0,0.061851,0.066341,0.000000,0.159989,0.061851,0.324169,0.010053,0.011314,0.000000,0.000000,0.798216,0.786388,1.104154,0.492597,-0.101684,0.197494,0.252642,0.126988,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
1,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,0.2,0.406057,0.675459,0.000000,0.510503,0.406057,0.395509,0.112243,0.001061,0.000000,0.000000,0.663455,0.726806,0.909532,0.479087,0.124476,0.114294,0.150240,0.174173,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
2,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,0.4,0.504598,0.573313,0.385832,0.557829,0.504598,0.437455,0.086062,0.177978,0.088863,0.182905,0.628389,0.720365,0.960963,0.522927,0.486604,0.042837,0.065167,0.217835,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
3,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,0.6,0.609531,0.806499,0.385832,0.557829,0.609531,0.493817,0.226866,0.009168,0.088863,0.182905,0.547500,0.591973,0.623166,0.419213,-0.025444,0.086849,-0.240452,0.175477,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
4,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,0.8,0.605239,0.796960,0.385832,0.557829,0.605239,0.453791,0.265056,0.038585,0.088863,0.182905,0.440232,0.492757,0.999941,0.369853,-0.467637,0.133200,-0.554436,0.160109,0.012898,0.0,0.111251,0.0,0.0,0.0,0.058205,0.0,0.111111,0.0,0.0,0.0
5,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,1.0,0.343659,0.391650,0.231449,0.432046,0.343659,0.420553,0.084473,0.003937,0.195448,0.175573,0.378629,0.315106,0.740326,0.391191,-0.423777,0.147335,-0.042812,0.142754,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
6,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,1.2,0.205929,0.086724,0.231449,0.429483,0.205929,0.350288,0.027559,0.000475,0.195448,0.175573,0.370514,0.265294,0.583295,0.342719,-0.168334,0.142988,0.020620,0.033202,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
7,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,1.4,0.338406,0.431286,0.231449,0.316600,0.338406,0.243089,0.101180,0.000232,0.195448,0.175573,0.195593,0.348042,0.469978,0.323244,-0.074320,0.052534,0.070646,0.025140,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
8,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,1.6,0.258206,0.433079,0.000000,0.316600,0.258206,0.243869,0.019700,0.016157,0.000000,0.000000,0.335049,0.369942,0.337788,0.266373,0.059975,0.143853,-0.083826,0.100117,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
9,001_後進時にトラックに衝突,anomaly,outdoor,anomaly/outdoor/001,1.8,0.069243,0.089799,0.000000,0.144166,0.069243,0.258814,0.003425,0.020364,0.000000,0.000000,0.457369,0.369039,0.402921,0.303387,0.071831,0.178614,-0.192252,0.172245,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0


saved summary            : /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/target_video_scores.csv
saved top windows        : /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/target_top_windows.csv
saved key metrics        : /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/all_target_key_metrics.csv
saved score overlay      : /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/overlay_anomaly_scores.png
saved feature overlay    : /workspaces/forklift/outputs/anomaly_feature_poc/target_inference/_comparison/overlay_key_metrics.png


## 6. ピーク窓の簡易確認

In [111]:
def top_contributing_features(model_name: str, row_index: int, model_inputs: dict[str, np.ndarray], top_n: int = 10) -> pd.DataFrame:
    artifact = score_model_artifacts[model_name]
    values = model_inputs[model_name][row_index]
    feature_names = np.asarray(artifact["feature_names"])
    order = np.argsort(np.abs(values))[::-1][:top_n]
    return pd.DataFrame({
        "model_name": model_name,
        "feature": feature_names[order],
        "group": [artifact.get("expanded_feature_group_map", {}).get(str(feature_names[i]), "other") for i in order],
        "scaled_value": values[order],
        "abs_scaled_value": np.abs(values[order]),
    })


if RUN_INFERENCE:
    for sample_id, result in target_results.items():
        scored_df = result["scored_df"]
        if scored_df is None or scored_df.empty:
            continue
        peak_idx = int(scored_df["anomaly_score"].idxmax())
        peak_pos = scored_df.index.get_loc(peak_idx)
        print("=" * 80)
        show_cols = ["video_id", "target_environment", "time", *[c for c in PLOT_SCORE_COLUMNS if c in scored_df.columns], "final_anomaly_score", "anomaly_score_smooth"]
        show_cols = list(dict.fromkeys(show_cols))
        print(scored_df.loc[peak_idx, show_cols])
        for model_name in score_model_artifacts:
            display(top_contributing_features(model_name, peak_pos, result["model_inputs"], top_n=12))
else:
    print("Top contributing features skipped because RUN_INFERENCE=False.")


video_id                001_後進時にトラックに衝突
target_environment              outdoor
time                                8.8
anomaly_score                  0.965875
audio_anomaly_score                 1.0
motion_anomaly_score           0.902501
sync_score                          1.0
final_anomaly_score            0.965875
anomaly_score_smooth           0.820947
Name: 44, dtype: object


,model_name,feature,group,scaled_value,abs_scaled_value
0,audio,audio_energy,audio_basic,5.506913,5.506913
1,audio,audio_rms,audio_basic,4.348166,4.348166
2,audio,audio_mel_15,audio_mel,3.448396,3.448396
3,audio,audio_ptp,audio_basic,3.409394,3.409394
4,audio,audio_peak,audio_basic,3.367608,3.367608
5,audio,audio_mel_14,audio_mel,3.263864,3.263864
6,audio,audio_bandwidth,audio_basic,3.157467,3.157467
7,audio,audio_mel_13,audio_mel,3.006707,3.006707
8,audio,audio_mel_12,audio_mel,2.774188,2.774188
9,audio,audio_high_freq_energy,audio_basic,2.521403,2.521403


,model_name,feature,group,scaled_value,abs_scaled_value
0,motion,front_broad_vibration_score,broad_vibration,2.889643,2.889643
1,motion,rear_broad_vibration_score,broad_vibration,2.016917,2.016917


video_id                005_後進旋回時トラックに衝突
target_environment               outdoor
time                                 9.2
anomaly_score                   0.838915
audio_anomaly_score             0.759639
motion_anomaly_score            0.937895
sync_score                      0.844074
final_anomaly_score             0.838915
anomaly_score_smooth            0.697801
Name: 46, dtype: object


,model_name,feature,group,scaled_value,abs_scaled_value
0,audio,audio_mel_15,audio_mel,3.133217,3.133217
1,audio,audio_mel_14,audio_mel,2.823779,2.823779
2,audio,audio_bandwidth,audio_basic,2.813241,2.813241
3,audio,audio_mel_13,audio_mel,2.706487,2.706487
4,audio,audio_mel_12,audio_mel,2.509251,2.509251
5,audio,audio_mel_11,audio_mel,1.948212,1.948212
6,audio,audio_mel_08,audio_mel,1.884454,1.884454
7,audio,audio_mel_09,audio_mel,1.851860,1.851860
8,audio,audio_mel_05,audio_mel,1.823310,1.823310
9,audio,audio_mel_10,audio_mel,1.647721,1.647721


...

,model_name,feature,group,scaled_value,abs_scaled_value
0,audio,audio_energy,audio_basic,6.676384,6.676384
1,audio,audio_rms,audio_basic,4.834917,4.834917
2,audio,audio_ptp,audio_basic,3.396371,3.396371
3,audio,audio_peak,audio_basic,3.284098,3.284098
4,audio,audio_mel_15,audio_mel,2.927813,2.927813
5,audio,audio_mel_14,audio_mel,2.541345,2.541345
6,audio,audio_mel_13,audio_mel,2.389715,2.389715
7,audio,audio_mel_12,audio_mel,2.122780,2.122780
8,audio,audio_mel_08,audio_mel,1.958936,1.958936
9,audio,audio_mel_11,audio_mel,1.728866,1.728866


,model_name,feature,group,scaled_value,abs_scaled_value
0,motion,front_broad_vibration_score,broad_vibration,1.962141,1.962141
1,motion,rear_broad_vibration_score,broad_vibration,-0.212254,0.212254


video_id                    1036
target_environment        indoor
time                         0.0
anomaly_score           0.534839
audio_anomaly_score     0.839851
motion_anomaly_score    0.151572
sync_score              0.519277
final_anomaly_score     0.534839
anomaly_score_smooth     0.34581
Name: 0, dtype: object


,model_name,feature,group,scaled_value,abs_scaled_value
0,audio,audio_mel_07,audio_mel,-2.552685,2.552685
1,audio,audio_mel_04,audio_mel,-2.500235,2.500235
2,audio,audio_mel_03,audio_mel,-2.500039,2.500039
3,audio,audio_mel_02,audio_mel,-2.497687,2.497687
4,audio,audio_mel_06,audio_mel,-2.371636,2.371636
5,audio,audio_mel_05,audio_mel,-2.364789,2.364789
6,audio,audio_mel_08,audio_mel,-2.354224,2.354224
7,audio,audio_mel_01,audio_mel,-2.265814,2.265814
8,audio,audio_mel_00,audio_mel,-2.075734,2.075734
9,audio,audio_centroid,audio_basic,1.872958,1.872958


,model_name,feature,group,scaled_value,abs_scaled_value
0,motion,rear_broad_vibration_score,broad_vibration,-0.933474,0.933474
1,motion,front_broad_vibration_score,broad_vibration,-0.130962,0.130962
